In [ ]:

# Environment setup, output folders, reproducibility, logging

import os
import sys
import json
import time
import random
import hashlib
import warnings
import platform
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


# Optional deep learning / GPU libraries

TORCH_AVAILABLE = False
CUDA_AVAILABLE = False

try:
    import torch
    TORCH_AVAILABLE = True
    CUDA_AVAILABLE = torch.cuda.is_available()
except Exception:
    torch = None


# Global settings

EXPERIMENT_NAME = "multimodal_surveillance_fusion"
SEED = 42
REPRO_MODE = True
START_TIME = time.time()
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID = f"run_{RUN_TIMESTAMP}_seed{SEED}"


# Output folders

BASE_DIR = Path("./outputs")
RUN_DIR = BASE_DIR / RUN_ID

FIG_DIR = RUN_DIR / "figures"
TAB_DIR = RUN_DIR / "tables"
OTH_DIR = RUN_DIR / "others"
LOG_DIR = RUN_DIR / "logs"
MODEL_DIR = RUN_DIR / "models"
GRAPH_DIR = RUN_DIR / "graphs"
CASE_DIR = RUN_DIR / "case_studies"

SUMMARY_FILE = LOG_DIR / "outputs_summary.txt"
CONFIG_FILE = LOG_DIR / "run_config.json"
ENV_FILE = LOG_DIR / "environment.json"
MANIFEST_FILE = LOG_DIR / "artifact_manifest.json"
SESSION_FILE = LOG_DIR / "session_metadata.json"

for d in [BASE_DIR, RUN_DIR, FIG_DIR, TAB_DIR, OTH_DIR, LOG_DIR, MODEL_DIR, GRAPH_DIR, CASE_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# Reproducibility

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Avoid globally silencing all warnings during development.
warnings.filterwarnings("default")

if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    if CUDA_AVAILABLE:
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    if REPRO_MODE:
        try:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        except Exception:
            pass

        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception:
            pass

DEVICE = "cuda" if (TORCH_AVAILABLE and CUDA_AVAILABLE) else "cpu"


# Utility functions

def append_summary(text: str, also_print: bool = False):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {text}"
    with open(SUMMARY_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")
    if also_print:
        print(line)

def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, ensure_ascii=False, default=str)

def load_json(path: Path, default):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default

def sha256_of_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def sha256_of_file(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def register_artifact(path: Path, category: str, description: str, extra: dict = None):
    manifest = load_json(MANIFEST_FILE, [])

    record = {
        "path": str(path),
        "category": category,
        "description": description,
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "exists": path.exists(),
        "sha256": sha256_of_file(path) if path.exists() else None
    }
    if extra is not None:
        record["extra"] = extra

    # Prevent duplicate entries for the same path + category + description
    duplicate_exists = any(
        (item.get("path") == record["path"]) and
        (item.get("category") == record["category"]) and
        (item.get("description") == record["description"])
        for item in manifest
    )

    if not duplicate_exists:
        manifest.append(record)
        save_json(manifest, MANIFEST_FILE)

def print_section(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def seconds_since_start() -> float:
    return time.time() - START_TIME

def get_package_version(pkg_name: str):
    try:
        module = importlib.import_module(pkg_name)
        return getattr(module, "__version__", "unknown")
    except Exception:
        return None


# Package versions

tracked_packages = {
    "numpy": get_package_version("numpy"),
    "pandas": get_package_version("pandas"),
    "torch": get_package_version("torch"),
    "sklearn": get_package_version("sklearn"),
    "matplotlib": get_package_version("matplotlib"),
    "seaborn": get_package_version("seaborn"),
    "scipy": get_package_version("scipy"),
    "networkx": get_package_version("networkx"),
    "cv2": get_package_version("cv2"),
    "PIL": get_package_version("PIL"),
    "shap": get_package_version("shap"),
    "tqdm": get_package_version("tqdm")
}


# Environment metadata

env_info = {
    "experiment_name": EXPERIMENT_NAME,
    "run_id": RUN_ID,
    "run_timestamp": RUN_TIMESTAMP,
    "seed": SEED,
    "repro_mode": REPRO_MODE,
    "device": DEVICE,
    "python_version_full": sys.version,
    "python_version_short": sys.version.split()[0],
    "platform": platform.platform(),
    "system": platform.system(),
    "system_release": platform.release(),
    "system_version": platform.version(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "python_executable": sys.executable,
    "working_directory": str(Path.cwd()),
    "package_versions": tracked_packages,
    "torch_available": TORCH_AVAILABLE,
    "cuda_available": CUDA_AVAILABLE
}


# GPU information

if TORCH_AVAILABLE:
    env_info["torch_version"] = torch.__version__

    try:
        env_info["mps_available"] = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    except Exception:
        env_info["mps_available"] = False

    if CUDA_AVAILABLE:
        try:
            gpu_count = torch.cuda.device_count()
            env_info["gpu_count"] = gpu_count
            env_info["cuda_version"] = getattr(torch.version, "cuda", None)
            env_info["cudnn_version"] = torch.backends.cudnn.version()

            gpu_devices = []
            for i in range(gpu_count):
                props = torch.cuda.get_device_properties(i)
                gpu_devices.append({
                    "device_index": i,
                    "device_name": torch.cuda.get_device_name(i),
                    "total_memory_GB": round(props.total_memory / (1024 ** 3), 2),
                    "multi_processor_count": getattr(props, "multi_processor_count", None),
                    "major": getattr(props, "major", None),
                    "minor": getattr(props, "minor", None)
                })

            env_info["gpu_devices"] = gpu_devices

        except Exception as e:
            env_info["gpu_info_error"] = str(e)


# Run configuration

run_config = {
    "experiment_name": EXPERIMENT_NAME,
    "seed": SEED,
    "repro_mode": REPRO_MODE,
    "run_id": RUN_ID,
    "run_timestamp": RUN_TIMESTAMP,
    "device": DEVICE,
    "base_dir": str(BASE_DIR),
    "run_dir": str(RUN_DIR),
    "figure_dir": str(FIG_DIR),
    "table_dir": str(TAB_DIR),
    "other_dir": str(OTH_DIR),
    "log_dir": str(LOG_DIR),
    "model_dir": str(MODEL_DIR),
    "graph_dir": str(GRAPH_DIR),
    "case_study_dir": str(CASE_DIR),
    "summary_file": str(SUMMARY_FILE),
    "environment_file": str(ENV_FILE),
    "manifest_file": str(MANIFEST_FILE)
}

session_metadata = {
    "start_datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "unix_start_time": START_TIME,
    "run_id": RUN_ID,
    "seed": SEED,
    "experiment_hash": sha256_of_text(EXPERIMENT_NAME),
    "folder_structure": {
        "base": str(BASE_DIR),
        "run": str(RUN_DIR),
        "figures": str(FIG_DIR),
        "tables": str(TAB_DIR),
        "others": str(OTH_DIR),
        "logs": str(LOG_DIR),
        "models": str(MODEL_DIR),
        "graphs": str(GRAPH_DIR),
        "case_studies": str(CASE_DIR)
    }
}


# Save metadata files

save_json(env_info, ENV_FILE)
save_json(run_config, CONFIG_FILE)
save_json(session_metadata, SESSION_FILE)

if not MANIFEST_FILE.exists():
    save_json([], MANIFEST_FILE)

register_artifact(ENV_FILE, "log", "Environment metadata")
register_artifact(CONFIG_FILE, "log", "Run configuration")
register_artifact(SESSION_FILE, "log", "Session metadata")


# Log start

append_summary("=" * 100)
append_summary("Notebook started")
append_summary(f"Experiment name: {EXPERIMENT_NAME}")
append_summary(f"Run ID: {RUN_ID}")
append_summary(f"Seed: {SEED}")
append_summary(f"Reproducibility mode: {REPRO_MODE}")
append_summary(f"Device: {DEVICE}")
append_summary(f"Working directory: {Path.cwd()}")
append_summary(f"Python version: {sys.version.split()[0]}")
append_summary(f"NumPy version: {np.__version__}")
append_summary(f"Pandas version: {pd.__version__}")
append_summary(f"PyTorch available: {TORCH_AVAILABLE}")
append_summary(f"CUDA available: {CUDA_AVAILABLE}")

if TORCH_AVAILABLE:
    append_summary(f"PyTorch version: {torch.__version__}")
    if CUDA_AVAILABLE:
        append_summary(f"CUDA version: {getattr(torch.version, 'cuda', None)}")
        append_summary(f"cuDNN version: {torch.backends.cudnn.version()}")


# Print summary

print_section("INITIALIZATION")
print(f"Experiment Name            : {EXPERIMENT_NAME}")
print(f"Run ID                     : {RUN_ID}")
print(f"Run Timestamp              : {RUN_TIMESTAMP}")
print(f"Seed                       : {SEED}")
print(f"Reproducibility Mode       : {REPRO_MODE}")
print(f"Device                     : {DEVICE}")

print_section("DIRECTORIES")
print(f"Base Directory             : {BASE_DIR.resolve()}")
print(f"Run Directory              : {RUN_DIR.resolve()}")
print(f"Figures Directory          : {FIG_DIR.resolve()}")
print(f"Tables Directory           : {TAB_DIR.resolve()}")
print(f"Others Directory           : {OTH_DIR.resolve()}")
print(f"Logs Directory             : {LOG_DIR.resolve()}")
print(f"Models Directory           : {MODEL_DIR.resolve()}")
print(f"Graphs Directory           : {GRAPH_DIR.resolve()}")
print(f"Case Studies Directory     : {CASE_DIR.resolve()}")

print_section("ENVIRONMENT")
print(f"Python Version             : {sys.version}")
print(f"Python Executable          : {sys.executable}")
print(f"Platform                   : {platform.platform()}")
print(f"System                     : {platform.system()}")
print(f"Release                    : {platform.release()}")
print(f"Machine                    : {platform.machine()}")
print(f"Processor                  : {platform.processor()}")
print(f"Working Directory          : {Path.cwd()}")

print_section("PACKAGE VERSIONS")
for k, v in tracked_packages.items():
    print(f"{k:<28}: {v}")

print_section("TORCH / GPU")
print(f"PyTorch Available          : {TORCH_AVAILABLE}")
if TORCH_AVAILABLE:
    print(f"PyTorch Version            : {torch.__version__}")
    print(f"CUDA Available             : {CUDA_AVAILABLE}")
    if CUDA_AVAILABLE:
        print(f"CUDA Version               : {getattr(torch.version, 'cuda', None)}")
        print(f"cuDNN Version              : {torch.backends.cudnn.version()}")
        print(f"GPU Count                  : {torch.cuda.device_count()}")
        for i, g in enumerate(env_info.get("gpu_devices", [])):
            print(f"GPU {i} Name               : {g['device_name']}")
            print(f"GPU {i} Total Memory (GB)  : {g['total_memory_GB']}")
            print(f"GPU {i} SM Count           : {g['multi_processor_count']}")
            print(f"GPU {i} Compute Capability : {g['major']}.{g['minor']}")
    else:
        print("CUDA not available. CPU execution will be used unless another accelerator is configured.")
else:
    print("PyTorch not installed.")

print_section("REPRODUCIBILITY")
print(f"PYTHONHASHSEED             : {os.environ.get('PYTHONHASHSEED')}")
print(f"random seed                : {SEED}")
print(f"numpy seed                 : {SEED}")
if TORCH_AVAILABLE:
    print(f"torch.manual_seed          : {SEED}")
    if CUDA_AVAILABLE:
        print(f"torch.cuda.manual_seed_all : {SEED}")
    try:
        print(f"cuDNN deterministic        : {torch.backends.cudnn.deterministic}")
        print(f"cuDNN benchmark            : {torch.backends.cudnn.benchmark}")
    except Exception:
        pass

print_section("FILES")
print(f"Summary File               : {SUMMARY_FILE.resolve()}")
print(f"Config File                : {CONFIG_FILE.resolve()}")
print(f"Environment File           : {ENV_FILE.resolve()}")
print(f"Session File               : {SESSION_FILE.resolve()}")
print(f"Manifest File              : {MANIFEST_FILE.resolve()}")

print_section("STATUS")
print("Environment setup completed successfully.")


INITIALIZATION
Experiment Name            : multimodal_surveillance_fusion
Run ID                     : run_20260409_172935_seed42
Run Timestamp              : 20260409_172935
Seed                       : 42
Reproducibility Mode       : True
Device                     : cuda

DIRECTORIES
Base Directory             : F:\MBZUAI UAE\7-CMES-SI\outputs
Run Directory              : F:\MBZUAI UAE\7-CMES-SI\outputs\run_20260409_172935_seed42
Figures Directory          : F:\MBZUAI UAE\7-CMES-SI\outputs\run_20260409_172935_seed42\figures
Tables Directory           : F:\MBZUAI UAE\7-CMES-SI\outputs\run_20260409_172935_seed42\tables
Others Directory           : F:\MBZUAI UAE\7-CMES-SI\outputs\run_20260409_172935_seed42\others
Logs Directory             : F:\MBZUAI UAE\7-CMES-SI\outputs\run_20260409_172935_seed42\logs
Models Directory           : F:\MBZUAI UAE\7-CMES-SI\outputs\run_20260409_172935_seed42\models
Graphs Directory           : F:\MBZUAI UAE\7-CMES-SI\outputs\run_20260409_172935_seed42

In [ ]:

# Core imports for analytics, modeling, explainability, graphs


import gc
import math
import pickle

import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import shap
import sklearn

from scipy import sparse, stats

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve

append_summary("Core libraries imported successfully.")

print("\n" + "=" * 80)
print("IMPORTS")
print("=" * 80)
print(f"matplotlib version         : {matplotlib.__version__}")
print(f"networkx version           : {nx.__version__}")
print(f"shap version               : {shap.__version__}")
print(f"scikit-learn version       : {sklearn.__version__}")
print(f"scipy sparse available     : {hasattr(sparse, 'csr_matrix')}")
print("Imports completed successfully.")


IMPORTS
matplotlib version         : 3.10.8
networkx version           : 3.4.2
shap version               : 0.49.1
scikit-learn version       : 1.7.2
scipy sparse available     : True
Imports completed successfully.


In [ ]:

# Experiment configuration


CONFIG = {
    # -----------------------------
    # Input files
    # -----------------------------
    "cyber_csv_path": "./datasets/ton-iot-train_test_network.csv",
    "video_csv_path": "./datasets/video_events_simulated.csv",

    # -----------------------------
    # Sampling / row limits
    # -----------------------------
    "max_cyber_rows": None,
    "max_video_rows": None,

    # -----------------------------
    # Split settings
    # -----------------------------
    "test_size": 0.30,
    "random_state": SEED,
    "stratify_cyber": True,
    "stratify_video": False,

    # -----------------------------
    # Core columns
    # -----------------------------
    "cyber_label_col": "label",
    "video_label_col": "label",
    "cyber_time_col": "timestamp",
    "video_time_col": "timestamp",

    # Optional entity / alignment columns
    "cyber_src_id_col": "src_ip",
    "cyber_dst_id_col": "dst_ip",
    "video_entity_col": "camera_id",
    "video_location_col": "location",

    # -----------------------------
    # Fusion settings
    # -----------------------------
    "fusion_time_window_sec": 120,
    "allow_missing_modalities": True,
    "fusion_strategy": "weighted_mean",

    # -----------------------------
    # Detection thresholds
    # -----------------------------
    "cyber_alert_threshold": 0.60,
    "video_alert_threshold": 0.60,
    "fusion_risk_threshold": 0.65,

    # -----------------------------
    # Uncertainty settings
    # -----------------------------
    "uncertainty_margin": 0.10,
    "use_uncertainty_filter": True,
    "uncertainty_method": "margin",

    # -----------------------------
    # Campaign / graph settings
    # -----------------------------
    "min_campaign_size": 2,
    "build_temporal_graph": True,
    "graph_directed": True,
    "merge_events_within_window": True,

    # -----------------------------
    # Explainability
    # -----------------------------
    "compute_shap": True,
    "max_shap_samples": 200,

    # -----------------------------
    # Output settings
    # -----------------------------
    "save_figures": True,
    "save_tables": True,
    "save_models": True,
    "figure_dpi": 300
}


# Basic config validation

assert 0 < CONFIG["test_size"] < 1, "test_size must be between 0 and 1"
assert 0 <= CONFIG["cyber_alert_threshold"] <= 1, "cyber_alert_threshold must be between 0 and 1"
assert 0 <= CONFIG["video_alert_threshold"] <= 1, "video_alert_threshold must be between 0 and 1"
assert 0 <= CONFIG["fusion_risk_threshold"] <= 1, "fusion_risk_threshold must be between 0 and 1"
assert 0 <= CONFIG["uncertainty_margin"] <= 1, "uncertainty_margin must be between 0 and 1"
assert CONFIG["fusion_time_window_sec"] > 0, "fusion_time_window_sec must be > 0"
assert CONFIG["min_campaign_size"] >= 1, "min_campaign_size must be >= 1"
assert CONFIG["fusion_strategy"] in {"weighted_mean"}, "Unsupported fusion strategy"
assert CONFIG["uncertainty_method"] in {"margin"}, "Unsupported uncertainty method"


# Update saved run configuration

run_config_full = load_json(CONFIG_FILE, {})
run_config_full.update(CONFIG)
save_json(run_config_full, CONFIG_FILE)

register_artifact(
    CONFIG_FILE,
    "log",
    "Updated run configuration",
    extra={
        "run_id": RUN_ID,
        "keys_added": list(CONFIG.keys())
    }
)

append_summary("Configuration loaded")
append_summary(f"Configuration attached to run_id={RUN_ID}")
append_summary(json.dumps(CONFIG, indent=2))

print("\n" + "=" * 80)
print("CONFIGURATION")
print("=" * 80)
print(json.dumps(CONFIG, indent=2))
print("\nConfiguration checks passed.")
print(f"Active run ID              : {RUN_ID}")
print(f"Configuration file updated : {CONFIG_FILE.resolve()}")


CONFIGURATION
{
  "cyber_csv_path": "./datasets/ton-iot-train_test_network.csv",
  "video_csv_path": "./datasets/video_events_simulated.csv",
  "max_cyber_rows": null,
  "max_video_rows": null,
  "test_size": 0.3,
  "random_state": 42,
  "stratify_cyber": true,
  "stratify_video": false,
  "cyber_label_col": "label",
  "video_label_col": "label",
  "cyber_time_col": "timestamp",
  "video_time_col": "timestamp",
  "cyber_src_id_col": "src_ip",
  "cyber_dst_id_col": "dst_ip",
  "video_entity_col": "camera_id",
  "video_location_col": "location",
  "fusion_time_window_sec": 120,
  "allow_missing_modalities": true,
  "fusion_strategy": "weighted_mean",
  "cyber_alert_threshold": 0.6,
  "video_alert_threshold": 0.6,
  "fusion_risk_threshold": 0.65,
  "uncertainty_margin": 0.1,
  "use_uncertainty_filter": true,
  "uncertainty_method": "margin",
  "min_campaign_size": 2,
  "build_temporal_graph": true,
  "graph_directed": true,
  "merge_events_within_window": true,
  "compute_shap": true,
  

In [ ]:

# Utility functions


def save_df(df, filename, index=False, category="table", description=None):
    path = TAB_DIR / filename
    existed_before = path.exists()

    df.to_csv(path, index=index)

    msg = f"Saved table: {path}"
    if existed_before:
        msg += " (overwritten)"

    print(msg)
    append_summary(msg)

    register_artifact(
        path=path,
        category=category,
        description=description or f"Table saved as {filename}",
        extra={
            "rows": int(df.shape[0]),
            "columns": int(df.shape[1]),
            "index_saved": index
        }
    )
    return path

def save_json_file(obj, filename, category="json", description=None):
    path = OTH_DIR / filename
    existed_before = path.exists()

    save_json(obj, path)

    msg = f"Saved json: {path}"
    if existed_before:
        msg += " (overwritten)"

    print(msg)
    append_summary(msg)

    register_artifact(
        path=path,
        category=category,
        description=description or f"JSON saved as {filename}"
    )
    return path

def save_figure(fig, filename, dpi=300, close=True, category="figure", description=None):
    path = FIG_DIR / filename
    existed_before = path.exists()

    try:
        fig.tight_layout()
    except Exception:
        pass

    fig.savefig(path, bbox_inches="tight", dpi=dpi)

    msg = f"Saved figure: {path}"
    if existed_before:
        msg += " (overwritten)"

    print(msg)
    append_summary(msg)

    register_artifact(
        path=path,
        category=category,
        description=description or f"Figure saved as {filename}",
        extra={"dpi": dpi}
    )

    if close:
        plt.close(fig)

    return path

def log_metric_block(title, metrics_dict, print_metrics=True, round_digits=6):
    append_summary(f"--- {title} ---")

    if print_metrics:
        print("\n" + "-" * 80)
        print(title)
        print("-" * 80)

    for k, v in metrics_dict.items():
        if isinstance(v, (float, np.floating)):
            v_out = round(float(v), round_digits)
        else:
            v_out = v

        append_summary(f"{k}: {v_out}")

        if print_metrics:
            print(f"{k}: {v_out}")

def save_metrics_json(title, metrics_dict, filename):
    serializable_metrics = {}
    for k, v in metrics_dict.items():
        if isinstance(v, (np.integer,)):
            serializable_metrics[k] = int(v)
        elif isinstance(v, (np.floating, float)):
            serializable_metrics[k] = float(v)
        else:
            serializable_metrics[k] = v

    obj = {
        "title": title,
        "metrics": serializable_metrics,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return save_json_file(
        obj=obj,
        filename=filename,
        category="metrics",
        description=f"Metrics block for {title}"
    )

def save_pickle(obj, filename, category="model", description=None):
    path = MODEL_DIR / filename
    existed_before = path.exists()

    with open(path, "wb") as f:
        pickle.dump(obj, f)

    msg = f"Saved pickle: {path}"
    if existed_before:
        msg += " (overwritten)"

    print(msg)
    append_summary(msg)

    register_artifact(
        path=path,
        category=category,
        description=description or f"Pickle saved as {filename}"
    )
    return path

print("Utility functions ready.")
print("Available functions: save_df, save_json_file, save_figure, log_metric_block, save_metrics_json, save_pickle")

Utility functions ready.
Available functions: save_df, save_json_file, save_figure, log_metric_block, save_metrics_json, save_pickle


In [ ]:

# Load cyber dataset


print("Loading cyber dataset...")

cyber_path = Path(CONFIG["cyber_csv_path"])
assert cyber_path.exists(), f"Cyber dataset file not found: {cyber_path}"

file_size_mb = cyber_path.stat().st_size / (1024 ** 2)
print(f"Cyber dataset path         : {cyber_path.resolve()}")
print(f"Cyber dataset size (MB)    : {file_size_mb:.2f}")

cyber_df = pd.read_csv(cyber_path)

if CONFIG["max_cyber_rows"] is not None:
    cyber_df = cyber_df.iloc[:CONFIG["max_cyber_rows"]].copy()

# Preserve raw loaded copy before later preprocessing decisions
cyber_df_raw_for_fusion = cyber_df.copy()

print("Cyber dataset loaded.")
print(f"Cyber dataset shape        : {cyber_df.shape}")

mem_mb = cyber_df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Memory usage (MB)          : {mem_mb:.2f}")

dup_count = int(cyber_df.duplicated().sum())
dup_pct = round((dup_count / len(cyber_df)) * 100, 4) if len(cyber_df) > 0 else 0.0
print(f"Duplicate rows             : {dup_count}")
print(f"Duplicate percentage       : {dup_pct:.4f}")

missing_total = int(cyber_df.isna().sum().sum())
print(f"Total missing values       : {missing_total}")

print("\nFirst 5 rows:")
print(cyber_df.head())

print("\nColumn names:")
print(list(cyber_df.columns))

print("\nDtype summary:")
dtype_df = pd.DataFrame({
    "column": cyber_df.columns,
    "dtype": cyber_df.dtypes.astype(str).values,
    "missing_count": cyber_df.isna().sum().values,
    "missing_pct": (cyber_df.isna().mean().values * 100).round(4),
    "n_unique": cyber_df.nunique(dropna=False).values
})
print(dtype_df)


# Time column validation

configured_time_col = CONFIG.get("cyber_time_col", "timestamp")
cyber_time_col_exists = configured_time_col in cyber_df.columns

if cyber_time_col_exists:
    print(f"\nConfigured cyber time column found: {configured_time_col}")
else:
    surrogate_time_col = "event_index"
    cyber_df[surrogate_time_col] = np.arange(len(cyber_df), dtype=np.int64)
    cyber_df_raw_for_fusion[surrogate_time_col] = cyber_df[surrogate_time_col].values
    print(f"\nConfigured cyber time column not found: {configured_time_col}")
    print(f"Created surrogate ordering column      : {surrogate_time_col}")
    print("Note: this is a pseudo-temporal sequence based on row order, not a true timestamp.")


# Label distribution

label_col = CONFIG.get("cyber_label_col", "label")
if label_col in cyber_df.columns:
    label_counts = cyber_df[label_col].value_counts(dropna=False).sort_index()
    label_dist_df = label_counts.rename_axis(label_col).reset_index(name="count")
    label_dist_df["percentage"] = (label_dist_df["count"] / len(cyber_df) * 100).round(4)

    print(f"\nLabel distribution ({label_col}):")
    print(label_dist_df)
else:
    label_dist_df = None
    print(f"\nLabel column '{label_col}' not found in cyber dataset.")


# Save schema and label summaries

save_df(
    dtype_df,
    "cyber_schema_summary.csv",
    description="Cyber dataset schema summary with dtypes, missing counts, and unique counts"
)

if label_dist_df is not None:
    save_df(
        label_dist_df,
        "cyber_label_distribution.csv",
        description="Cyber dataset label distribution"
    )


# Save compact dataset load summary

cyber_load_summary = {
    "path": str(cyber_path.resolve()),
    "shape": {"rows": int(cyber_df.shape[0]), "columns": int(cyber_df.shape[1])},
    "file_size_mb": round(file_size_mb, 4),
    "memory_usage_mb": round(mem_mb, 4),
    "duplicate_rows": dup_count,
    "duplicate_percentage": dup_pct,
    "total_missing_values": missing_total,
    "column_names": list(cyber_df.columns),
    "configured_time_column": configured_time_col,
    "configured_time_column_exists": cyber_time_col_exists,
    "surrogate_time_column_created": None if cyber_time_col_exists else surrogate_time_col,
    "numeric_column_count": int((cyber_df.dtypes != "object").sum()),
    "object_column_count": int((cyber_df.dtypes == "object").sum())
}
save_json_file(
    cyber_load_summary,
    "cyber_load_summary.json",
    category="dataset_summary",
    description="Cyber dataset load summary"
)

append_summary(f"Cyber dataset path: {cyber_path.resolve()}")
append_summary(f"Cyber dataset shape: {cyber_df.shape}")
append_summary(f"Cyber dataset size (MB): {file_size_mb:.2f}")
append_summary(f"Cyber memory usage (MB): {mem_mb:.2f}")
append_summary(f"Cyber duplicate rows: {dup_count}")
append_summary(f"Cyber duplicate percentage: {dup_pct:.4f}")
append_summary(f"Cyber total missing values: {missing_total}")
append_summary(f"Cyber configured time column: {configured_time_col}")
append_summary(f"Cyber configured time column exists: {cyber_time_col_exists}")
if not cyber_time_col_exists:
    append_summary(f"Cyber surrogate time column created: {surrogate_time_col}")
append_summary(f"Cyber columns: {list(cyber_df.columns)}")

Loading cyber dataset...
Cyber dataset path         : F:\MBZUAI UAE\7-CMES-SI\datasets\ton-iot-train_test_network.csv
Cyber dataset size (MB)    : 28.52
Cyber dataset loaded.
Cyber dataset shape        : (211043, 44)
Memory usage (MB)          : 350.24
Duplicate rows             : 20569
Duplicate percentage       : 9.7464
Total missing values       : 0

First 5 rows:
          src_ip  src_port         dst_ip  dst_port proto service    duration  \
0   192.168.1.37      4444  192.168.1.193     49178   tcp       -  290.371539   
1  192.168.1.193     49180   192.168.1.37      8080   tcp       -    0.000102   
2  192.168.1.193     49180   192.168.1.37      8080   tcp       -    0.000148   
3  192.168.1.193     49180   192.168.1.37      8080   tcp       -    0.000113   
4  192.168.1.193     49180   192.168.1.37      8080   tcp       -    0.000130   

   src_bytes  dst_bytes conn_state  ...  http_response_body_len  \
0     101568       2592        OTH  ...                       0   
1        

In [ ]:

# Cyber dataset audit


audit_exclude_cols = ["event_index"] if "event_index" in cyber_df.columns else []
cyber_df_audit = cyber_df.drop(columns=audit_exclude_cols, errors="ignore").copy()

cyber_dups = int(cyber_df_audit.duplicated().sum())
cyber_dup_pct = round((cyber_dups / len(cyber_df_audit)) * 100, 4) if len(cyber_df_audit) > 0 else 0.0
cyber_missing_total = int(cyber_df_audit.isna().sum().sum())
cyber_missing_pct = round((cyber_missing_total / (cyber_df_audit.shape[0] * cyber_df_audit.shape[1])) * 100, 6) if cyber_df_audit.size > 0 else 0.0
cyber_mem_mb = round(cyber_df.memory_usage(deep=True).sum() / (1024 ** 2), 4)

constant_cols = [c for c in cyber_df_audit.columns if cyber_df_audit[c].nunique(dropna=False) <= 1]
object_cols = cyber_df_audit.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = cyber_df_audit.select_dtypes(include=[np.number]).columns.tolist()

n_rows = len(cyber_df_audit)
cardinality_df = pd.DataFrame({
    "column": cyber_df.columns,
    "n_unique": [cyber_df[c].nunique(dropna=False) for c in cyber_df.columns],
})
cardinality_df["unique_ratio"] = (cardinality_df["n_unique"] / len(cyber_df)).round(6)
cardinality_df = cardinality_df.sort_values(["unique_ratio", "n_unique"], ascending=False).reset_index(drop=True)

leakage_keywords = ["src", "dst", "ip", "port", "uri", "dns", "id", "timestamp", "time"]
possible_identifier_cols = [
    c for c in cyber_df_audit.columns
    if any(k in c.lower() for k in leakage_keywords)
]

audit_cyber_df = pd.DataFrame({
    "dataset": ["cyber"],
    "n_rows_original_view": [int(len(cyber_df_audit))],
    "n_cols_original_view": [int(cyber_df_audit.shape[1])],
    "n_cols_working_df": [int(cyber_df.shape[1])],
    "excluded_from_duplicate_audit": [",".join(audit_exclude_cols) if audit_exclude_cols else ""],
    "duplicates": [cyber_dups],
    "duplicate_pct": [cyber_dup_pct],
    "total_missing_values": [cyber_missing_total],
    "missing_pct_all_cells": [cyber_missing_pct],
    "memory_usage_mb_working_df": [cyber_mem_mb],
    "n_numeric_cols_original_view": [len(numeric_cols)],
    "n_object_cols_original_view": [len(object_cols)],
    "n_constant_cols_original_view": [len(constant_cols)],
    "n_possible_identifier_cols": [len(possible_identifier_cols)]
})

save_df(
    audit_cyber_df,
    "initial_cyber_data_audit.csv",
    description="Initial cyber dataset audit summary"
)

if len(constant_cols) > 0:
    constant_cols_df = pd.DataFrame({"column": constant_cols})
    save_df(
        constant_cols_df,
        "cyber_constant_columns.csv",
        description="Cyber dataset columns with one unique value"
    )

save_df(
    cardinality_df,
    "cyber_column_cardinality.csv",
    description="Cyber dataset column cardinality and unique-ratio summary"
)

possible_identifier_df = pd.DataFrame({"column": possible_identifier_cols})
save_df(
    possible_identifier_df,
    "cyber_possible_identifier_columns.csv",
    description="Cyber dataset columns flagged as possible identifiers or leakage-sensitive fields"
)

label_col = CONFIG.get("cyber_label_col", "label")
if label_col in cyber_df_audit.columns:
    cyber_label_dist = cyber_df_audit[label_col].value_counts(dropna=False).rename_axis(label_col).reset_index(name="count")
    cyber_label_dist["percentage"] = (cyber_label_dist["count"] / len(cyber_df_audit) * 100).round(4)

    save_df(
        cyber_label_dist,
        "cyber_label_distribution_audit.csv",
        description="Cyber dataset label distribution during audit"
    )
else:
    cyber_label_dist = None

append_summary(f"Cyber duplicate audit excluded columns: {audit_exclude_cols}")
append_summary(f"Cyber rows (original audit view): {len(cyber_df_audit)}")
append_summary(f"Cyber columns (original audit view): {cyber_df_audit.shape[1]}")
append_summary(f"Cyber columns (working df): {cyber_df.shape[1]}")
append_summary(f"Cyber duplicates: {cyber_dups}")
append_summary(f"Cyber duplicate percentage: {cyber_dup_pct}")
append_summary(f"Cyber total missing values: {cyber_missing_total}")
append_summary(f"Cyber memory usage working df (MB): {cyber_mem_mb}")
append_summary(f"Cyber possible identifier columns: {possible_identifier_cols}")

print(audit_cyber_df)

print("\nConstant columns:")
print(constant_cols if len(constant_cols) > 0 else "None")

print("\nTop 10 columns by unique ratio:")
print(cardinality_df.head(10))

print("\nPossible identifier / leakage-sensitive columns:")
print(possible_identifier_cols if len(possible_identifier_cols) > 0 else "None")

if cyber_label_dist is not None:
    print(f"\nLabel distribution ({label_col}):")
    print(cyber_label_dist)

print(f"\nActive run ID: {RUN_ID}")

Saved table: outputs\run_20260409_172935_seed42\tables\initial_cyber_data_audit.csv (overwritten)
Saved table: outputs\run_20260409_172935_seed42\tables\cyber_column_cardinality.csv (overwritten)
Saved table: outputs\run_20260409_172935_seed42\tables\cyber_possible_identifier_columns.csv (overwritten)
Saved table: outputs\run_20260409_172935_seed42\tables\cyber_label_distribution_audit.csv (overwritten)
  dataset  n_rows_original_view  n_cols_original_view  n_cols_working_df  \
0   cyber                211043                    44                 45   

  excluded_from_duplicate_audit  duplicates  duplicate_pct  \
0                   event_index       20569         9.7464   

   total_missing_values  missing_pct_all_cells  memory_usage_mb_working_df  \
0                     0                    0.0                    351.8465   

   n_numeric_cols_original_view  n_object_cols_original_view  \
0                            17                           27   

   n_constant_cols_original_v

In [ ]:

# Load video dataset or generate fallback dataset


video_path = Path(CONFIG["video_csv_path"])
video_source_type = None

required_video_cols = [
    "timestamp",
    "camera_id",
    "zone_id",
    "authenticity_score",
    "temporal_consistency",
    "uncertainty_score",
    "suspicious_label"
]

if video_path.exists():
    print("Loading video dataset...")
    video_df = pd.read_csv(video_path)

    if CONFIG["max_video_rows"] is not None:
        video_df = video_df.iloc[:CONFIG["max_video_rows"]].copy()

    video_source_type = "loaded_file"
    append_summary(f"Loaded video dataset from: {video_path}")

else:
    print("Video dataset not found. Generating fallback dataset...")

    n_samples = len(cyber_df)
    if "label" not in cyber_df.columns:
        raise RuntimeError("Cyber dataset must contain a 'label' column.")

    base_times = pd.date_range(start="2024-01-01", periods=n_samples, freq="s")

    video_df = pd.DataFrame({
        "timestamp": base_times,
        "camera_id": np.random.randint(1, 11, size=n_samples),
        "zone_id": np.random.randint(1, 6, size=n_samples),
        "authenticity_score": np.random.uniform(0.15, 1.0, size=n_samples),
        "temporal_consistency": np.random.uniform(0.45, 1.0, size=n_samples),
        "uncertainty_score": np.random.uniform(0.0, 0.45, size=n_samples)
    })

    attack_idx = cyber_df.index[cyber_df["label"] == 1].to_numpy()
    n_correlated = max(1, int(0.30 * len(attack_idx)))

    sampled_idx = np.random.choice(attack_idx, size=n_correlated, replace=False)
    video_df.loc[sampled_idx, "authenticity_score"] = np.random.uniform(0.0, 0.35, size=n_correlated)
    video_df.loc[sampled_idx, "temporal_consistency"] = np.random.uniform(0.35, 0.75, size=n_correlated)
    video_df.loc[sampled_idx, "uncertainty_score"] = np.random.uniform(0.0, 0.30, size=n_correlated)

    noise = np.random.uniform(0.0, 0.25, size=n_samples)
    latent_score = (
        0.55 * (1.0 - video_df["authenticity_score"].values) +
        0.30 * (1.0 - video_df["temporal_consistency"].values) +
        0.15 * video_df["uncertainty_score"].values +
        noise
    )

    video_df["suspicious_label"] = (latent_score >= np.quantile(latent_score, 0.55)).astype(int)

    video_path.parent.mkdir(parents=True, exist_ok=True)
    video_df.to_csv(video_path, index=False)

    video_source_type = "generated_fallback"
    append_summary("Generated fallback video dataset.")
    append_summary("Cross-modal correlation was injected during fallback generation.")
    append_summary(f"Saved generated video dataset to: {video_path}")


# Parse and validate

if "timestamp" in video_df.columns:
    video_df["timestamp"] = pd.to_datetime(video_df["timestamp"], errors="coerce")

missing_required = [c for c in required_video_cols if c not in video_df.columns]
if missing_required:
    raise ValueError(f"Video dataset is missing required columns: {missing_required}")

video_df_raw_for_fusion = video_df.copy()


# Basic audit

video_shape = video_df.shape
video_mem_mb = video_df.memory_usage(deep=True).sum() / (1024 ** 2)
video_dups = int(video_df.duplicated().sum())
video_missing_total = int(video_df.isna().sum().sum())
video_ts_min = str(video_df["timestamp"].min()) if "timestamp" in video_df.columns else None
video_ts_max = str(video_df["timestamp"].max()) if "timestamp" in video_df.columns else None
video_ts_monotonic = bool(video_df["timestamp"].is_monotonic_increasing) if "timestamp" in video_df.columns else None
video_camera_count = int(video_df["camera_id"].nunique()) if "camera_id" in video_df.columns else None
video_zone_count = int(video_df["zone_id"].nunique()) if "zone_id" in video_df.columns else None
row_count_matches_cyber = int(len(video_df)) == int(len(cyber_df))

video_dtype_df = pd.DataFrame({
    "column": video_df.columns,
    "dtype": video_df.dtypes.astype(str).values,
    "missing_count": video_df.isna().sum().values,
    "missing_pct": (video_df.isna().mean().values * 100).round(4),
    "n_unique": video_df.nunique(dropna=False).values
})

video_label_dist = None
if "suspicious_label" in video_df.columns:
    video_label_dist = video_df["suspicious_label"].value_counts(dropna=False).sort_index().rename_axis("suspicious_label").reset_index(name="count")
    video_label_dist["percentage"] = (video_label_dist["count"] / len(video_df) * 100).round(4)


# Save summaries

save_df(
    video_dtype_df,
    "video_schema_summary.csv",
    description="Video dataset schema summary"
)

if video_label_dist is not None:
    save_df(
        video_label_dist,
        "video_label_distribution.csv",
        description="Video dataset suspicious label distribution"
    )

video_load_summary = {
    "path": str(video_path.resolve()),
    "source_type": video_source_type,
    "shape": {"rows": int(video_shape[0]), "columns": int(video_shape[1])},
    "memory_usage_mb": round(video_mem_mb, 4),
    "duplicate_rows": video_dups,
    "total_missing_values": video_missing_total,
    "timestamp_min": video_ts_min,
    "timestamp_max": video_ts_max,
    "timestamp_monotonic_increasing": video_ts_monotonic,
    "unique_camera_count": video_camera_count,
    "unique_zone_count": video_zone_count,
    "row_count_matches_cyber": row_count_matches_cyber,
    "columns": list(video_df.columns)
}

save_json_file(
    video_load_summary,
    "video_load_summary.json",
    category="dataset_summary",
    description="Video dataset load summary"
)

append_summary(f"Video dataset source type: {video_source_type}")
append_summary(f"Video dataset shape: {video_shape}")
append_summary(f"Video duplicate rows: {video_dups}")
append_summary(f"Video total missing values: {video_missing_total}")
append_summary(f"Video timestamp min: {video_ts_min}")
append_summary(f"Video timestamp max: {video_ts_max}")
append_summary(f"Video timestamp monotonic increasing: {video_ts_monotonic}")
append_summary(f"Video unique cameras: {video_camera_count}")
append_summary(f"Video unique zones: {video_zone_count}")
append_summary(f"Video row count matches cyber: {row_count_matches_cyber}")


# Print

print("Video dataset ready.")
print(f"Video source type          : {video_source_type}")
print(f"Shape                      : {video_df.shape}")
print(f"Memory usage (MB)          : {video_mem_mb:.2f}")
print(f"Duplicate rows             : {video_dups}")
print(f"Total missing values       : {video_missing_total}")
print(f"Timestamp min              : {video_ts_min}")
print(f"Timestamp max              : {video_ts_max}")
print(f"Timestamp monotonic        : {video_ts_monotonic}")
print(f"Unique cameras             : {video_camera_count}")
print(f"Unique zones               : {video_zone_count}")
print(f"Row count matches cyber    : {row_count_matches_cyber}")
print(f"Active run ID              : {RUN_ID}")

print("\nFirst 5 rows:")
print(video_df.head())

print("\nDtype summary:")
print(video_dtype_df)

if video_label_dist is not None:
    print("\nSuspicious label distribution:")
    print(video_label_dist)

Loading video dataset...
Saved table: outputs\run_20260409_172935_seed42\tables\video_schema_summary.csv
Saved table: outputs\run_20260409_172935_seed42\tables\video_label_distribution.csv
Saved json: outputs\run_20260409_172935_seed42\others\video_load_summary.json
Video dataset ready.
Video source type          : loaded_file
Shape                      : (211043, 7)
Memory usage (MB)          : 11.27
Duplicate rows             : 0
Total missing values       : 0
Timestamp min              : 2024-01-01 00:00:00
Timestamp max              : 2024-05-26 13:22:00
Timestamp monotonic        : False
Unique cameras             : 10
Unique zones               : 5
Row count matches cyber    : True
Active run ID              : run_20260409_172935_seed42

First 5 rows:
            timestamp  camera_id  zone_id  authenticity_score  \
0 2024-04-13 16:35:00          7        5            0.120937   
1 2024-02-15 22:56:00          4        1            0.058880   
2 2024-01-13 22:50:00          8     

In [ ]:

# Video dataset audit


video_dups = int(video_df.duplicated().sum())
video_dup_pct = round((video_dups / len(video_df)) * 100, 4) if len(video_df) > 0 else 0.0
video_missing_total = int(video_df.isna().sum().sum())
video_missing_pct = round((video_missing_total / (video_df.shape[0] * video_df.shape[1])) * 100, 6) if video_df.size > 0 else 0.0
video_mem_mb = round(video_df.memory_usage(deep=True).sum() / (1024 ** 2), 4)

constant_cols = [c for c in video_df.columns if video_df[c].nunique(dropna=False) <= 1]
object_cols = video_df.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = video_df.select_dtypes(include=[np.number]).columns.tolist()
datetime_cols = video_df.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()

timestamp_col = CONFIG.get("video_time_col", "timestamp")
timestamp_valid_count = None
timestamp_min = None
timestamp_max = None
timestamp_monotonic = None

if timestamp_col in video_df.columns:
    ts = pd.to_datetime(video_df[timestamp_col], errors="coerce")
    timestamp_valid_count = int(ts.notna().sum())
    if ts.notna().any():
        timestamp_min = str(ts.min())
        timestamp_max = str(ts.max())
        timestamp_monotonic = bool(ts.is_monotonic_increasing)

configured_label_col = CONFIG.get("video_label_col", "label")
effective_label_col = configured_label_col if configured_label_col in video_df.columns else (
    "suspicious_label" if "suspicious_label" in video_df.columns else None
)

audit_video_df = pd.DataFrame({
    "dataset": ["video"],
    "n_rows": [int(len(video_df))],
    "n_cols": [int(video_df.shape[1])],
    "duplicates": [video_dups],
    "duplicate_pct": [video_dup_pct],
    "total_missing_values": [video_missing_total],
    "missing_pct_all_cells": [video_missing_pct],
    "memory_usage_mb": [video_mem_mb],
    "n_numeric_cols": [len(numeric_cols)],
    "n_object_cols": [len(object_cols)],
    "n_datetime_cols": [len(datetime_cols)],
    "n_constant_cols": [len(constant_cols)],
    "valid_timestamp_count": [timestamp_valid_count],
    "timestamp_min": [timestamp_min],
    "timestamp_max": [timestamp_max],
    "timestamp_monotonic": [timestamp_monotonic],
    "configured_label_col": [configured_label_col],
    "effective_label_col": [effective_label_col]
})

save_df(
    audit_video_df,
    "initial_video_data_audit.csv",
    description="Initial video dataset audit summary"
)

if len(constant_cols) > 0:
    constant_cols_df = pd.DataFrame({"column": constant_cols})
    save_df(
        constant_cols_df,
        "video_constant_columns.csv",
        description="Video dataset columns with one unique value"
    )

label_dist_df = None
if effective_label_col is not None:
    label_dist_df = video_df[effective_label_col].value_counts(dropna=False).rename_axis(effective_label_col).reset_index(name="count")
    label_dist_df["percentage"] = (label_dist_df["count"] / len(video_df) * 100).round(4)

    save_df(
        label_dist_df,
        "video_label_distribution_audit.csv",
        description="Video dataset label distribution during audit"
    )

append_summary(f"Video rows: {len(video_df)}")
append_summary(f"Video columns: {video_df.shape[1]}")
append_summary(f"Video duplicates: {video_dups}")
append_summary(f"Video duplicate percentage: {video_dup_pct}")
append_summary(f"Video total missing values: {video_missing_total}")
append_summary(f"Video memory usage (MB): {video_mem_mb}")
append_summary(f"Video numeric columns: {len(numeric_cols)}")
append_summary(f"Video object columns: {len(object_cols)}")
append_summary(f"Video datetime columns: {len(datetime_cols)}")
append_summary(f"Video constant columns: {len(constant_cols)}")
append_summary(f"Video timestamp min: {timestamp_min}")
append_summary(f"Video timestamp max: {timestamp_max}")
append_summary(f"Video timestamp monotonic: {timestamp_monotonic}")
append_summary(f"Video configured label column: {configured_label_col}")
append_summary(f"Video effective label column: {effective_label_col}")

print(audit_video_df)

print("\nConstant columns:")
print(constant_cols if len(constant_cols) > 0 else "None")

if label_dist_df is not None:
    print(f"\nLabel distribution ({effective_label_col}):")
    print(label_dist_df)

print(f"\nActive run ID: {RUN_ID}")

Saved table: outputs\run_20260409_172935_seed42\tables\initial_video_data_audit.csv
Saved table: outputs\run_20260409_172935_seed42\tables\video_label_distribution_audit.csv
  dataset  n_rows  n_cols  duplicates  duplicate_pct  total_missing_values  \
0   video  211043       7           0            0.0                     0   

   missing_pct_all_cells  memory_usage_mb  n_numeric_cols  n_object_cols  \
0                    0.0           11.271               6              0   

   n_datetime_cols  n_constant_cols  valid_timestamp_count  \
0                1                0                 211043   

         timestamp_min        timestamp_max  timestamp_monotonic  \
0  2024-01-01 00:00:00  2024-05-26 13:22:00                False   

  configured_label_col effective_label_col  
0                label    suspicious_label  

Constant columns:
None

Label distribution (suspicious_label):
   suspicious_label   count  percentage
0                 1  172982     81.9653
1                 0 

In [ ]:

# Cyber data cleaning and target definition


dedup_exclude_cols = ["event_index"] if "event_index" in cyber_df.columns else []

cyber_df_model = cyber_df.drop(columns=dedup_exclude_cols, errors="ignore").drop_duplicates().copy()

# Recreate surrogate ordering column after deduplication, if needed
if "event_index" in cyber_df.columns:
    cyber_df_model["event_index"] = np.arange(len(cyber_df_model), dtype=np.int64)

TARGET_CYBER = CONFIG.get("cyber_label_col", "label")

assert TARGET_CYBER in cyber_df_model.columns, f"{TARGET_CYBER} not found in cyber_df_model"
assert cyber_df_model[TARGET_CYBER].isna().sum() == 0, f"{TARGET_CYBER} contains missing values"

target_unique = sorted(cyber_df_model[TARGET_CYBER].dropna().unique().tolist())
assert set(target_unique).issubset({0, 1}), f"{TARGET_CYBER} must be binary with values in {{0,1}}, found: {target_unique}"

rows_before_dedup = int(len(cyber_df))
rows_after_dedup = int(len(cyber_df_model))
duplicates_removed = rows_before_dedup - rows_after_dedup


# Detect datetime columns

datetime_cols_cyber = []
failed_datetime_parse = []

for col in cyber_df_model.columns:
    if col == TARGET_CYBER:
        continue

    if pd.api.types.is_datetime64_any_dtype(cyber_df_model[col]):
        datetime_cols_cyber.append(col)
        continue

    col_lower = col.lower()
    looks_temporal = any(token in col_lower for token in ["time", "date", "timestamp", "datetime"])

    if looks_temporal:
        try:
            parsed = pd.to_datetime(cyber_df_model[col], errors="raise")
            cyber_df_model[col] = parsed
            datetime_cols_cyber.append(col)
        except Exception:
            failed_datetime_parse.append(col)


# Expand datetime features

for col in datetime_cols_cyber:
    cyber_df_model[f"{col}_hour"] = cyber_df_model[col].dt.hour
    cyber_df_model[f"{col}_minute"] = cyber_df_model[col].dt.minute
    cyber_df_model[f"{col}_second"] = cyber_df_model[col].dt.second
    cyber_df_model[f"{col}_dayofweek"] = cyber_df_model[col].dt.dayofweek
    cyber_df_model[f"{col}_unix"] = cyber_df_model[col].astype("int64") // 10**9

if len(datetime_cols_cyber) > 0:
    cyber_df_model = cyber_df_model.drop(columns=datetime_cols_cyber)


# Column roles

explicit_identifier_cols = [
    c for c in ["src_ip", "dst_ip", "event_index", "flow_id", "session_id", "uid"]
    if c in cyber_df_model.columns
]

analysis_only_cols = [c for c in ["type"] if c in cyber_df_model.columns]

exclude_from_model_cyber = sorted(set(explicit_identifier_cols + analysis_only_cols))

X_cyber_full = cyber_df_model.drop(columns=[TARGET_CYBER]).copy()
X_cyber_model = cyber_df_model.drop(columns=[TARGET_CYBER] + exclude_from_model_cyber, errors="ignore").copy()
y_cyber = cyber_df_model[TARGET_CYBER].astype(int).copy()

num_cols_cyber = X_cyber_model.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_cyber = X_cyber_model.select_dtypes(exclude=[np.number]).columns.tolist()


# Cleaning summary

cyber_cleaning_summary = {
    "target_column": TARGET_CYBER,
    "dedup_excluded_columns": dedup_exclude_cols,
    "rows_before_dedup": rows_before_dedup,
    "rows_after_dedup": rows_after_dedup,
    "duplicates_removed": duplicates_removed,
    "datetime_columns_detected": datetime_cols_cyber,
    "datetime_parse_failed_columns": failed_datetime_parse,
    "explicit_identifier_columns_excluded_from_model": explicit_identifier_cols,
    "analysis_only_columns_excluded_from_model": analysis_only_cols,
    "n_model_features_total": int(X_cyber_model.shape[1]),
    "n_model_numeric_features": int(len(num_cols_cyber)),
    "n_model_categorical_features": int(len(cat_cols_cyber)),
    "target_positive_ratio": float(y_cyber.mean()),
    "target_distribution": {str(k): int(v) for k, v in y_cyber.value_counts().sort_index().to_dict().items()}
}

save_json_file(
    cyber_cleaning_summary,
    "cyber_cleaning_summary.json",
    category="preprocessing",
    description="Cyber data cleaning and feature-role summary"
)

feature_role_df = pd.DataFrame({
    "column": X_cyber_full.columns,
    "used_for_model": [c in X_cyber_model.columns for c in X_cyber_full.columns],
    "is_numeric_in_model": [c in num_cols_cyber for c in X_cyber_full.columns],
    "is_categorical_in_model": [c in cat_cols_cyber for c in X_cyber_full.columns],
    "excluded_as_identifier_or_analysis": [c in exclude_from_model_cyber for c in X_cyber_full.columns]
})

save_df(
    feature_role_df,
    "cyber_feature_role_summary.csv",
    description="Cyber feature inclusion and exclusion summary"
)

append_summary(f"Cyber target: {TARGET_CYBER}")
append_summary(f"Cyber dedup excluded columns: {dedup_exclude_cols}")
append_summary(f"Cyber rows before dedup: {rows_before_dedup}")
append_summary(f"Cyber rows after dedup: {rows_after_dedup}")
append_summary(f"Cyber duplicates removed: {duplicates_removed}")
append_summary(f"Cyber datetime columns handled: {datetime_cols_cyber}")
append_summary(f"Cyber datetime parse failed columns: {failed_datetime_parse}")
append_summary(f"Cyber explicit identifier columns excluded from model: {explicit_identifier_cols}")
append_summary(f"Cyber analysis-only columns excluded from model: {analysis_only_cols}")
append_summary(f"Cyber positive ratio: {float(y_cyber.mean()):.6f}")
append_summary(f"Cyber model numeric columns: {len(num_cols_cyber)}")
append_summary(f"Cyber model categorical columns: {len(cat_cols_cyber)}")

print("Cyber target distribution:")
print(y_cyber.value_counts(normalize=True).sort_index())

print("\nCyber target counts:")
print(y_cyber.value_counts().sort_index())

print("\nDatetime columns handled:")
print(datetime_cols_cyber if len(datetime_cols_cyber) > 0 else "None")

print("\nColumns excluded from model:")
print(exclude_from_model_cyber if len(exclude_from_model_cyber) > 0 else "None")

print("\nModel feature matrix shape:")
print(X_cyber_model.shape)

print(f"\nNumeric model columns      : {len(num_cols_cyber)}")
print(f"Categorical model columns  : {len(cat_cols_cyber)}")
print(f"Active run ID              : {RUN_ID}")

Saved json: outputs\run_20260409_172935_seed42\others\cyber_cleaning_summary.json (overwritten)
Saved table: outputs\run_20260409_172935_seed42\tables\cyber_feature_role_summary.csv (overwritten)
Cyber target distribution:
label
0    0.220713
1    0.779287
Name: proportion, dtype: float64

Cyber target counts:
label
0     42040
1    148434
Name: count, dtype: int64

Datetime columns handled:
None

Columns excluded from model:
['dst_ip', 'event_index', 'src_ip', 'type']

Model feature matrix shape:
(190474, 40)

Numeric model columns      : 16
Categorical model columns  : 24
Active run ID              : run_20260409_172935_seed42


In [ ]:

# Video data cleaning and target definition


video_df_model = video_df.drop_duplicates().copy()

candidate_targets = [
    CONFIG.get("video_label_col", None),
    "suspicious_label",
    "label"
]
candidate_targets = [c for c in candidate_targets if c is not None]

TARGET_VIDEO = None
for c in candidate_targets:
    if c in video_df_model.columns:
        TARGET_VIDEO = c
        break

assert TARGET_VIDEO is not None, (
    f"No valid video target column found. Tried: {candidate_targets}. "
    f"Available columns: {list(video_df_model.columns)}"
)

CONFIG["video_label_col"] = TARGET_VIDEO

# Persist corrected config
run_config_full = load_json(CONFIG_FILE, {})
run_config_full["video_label_col"] = TARGET_VIDEO
save_json(run_config_full, CONFIG_FILE)

assert video_df_model[TARGET_VIDEO].isna().sum() == 0, f"{TARGET_VIDEO} contains missing values"

target_unique = sorted(video_df_model[TARGET_VIDEO].dropna().unique().tolist())
assert set(target_unique).issubset({0, 1}), (
    f"{TARGET_VIDEO} must be binary with values in {{0,1}}, found: {target_unique}"
)

rows_before_dedup = int(len(video_df))
rows_after_dedup = int(len(video_df_model))
duplicates_removed = rows_before_dedup - rows_after_dedup


# Detect datetime columns

datetime_cols_video = []
failed_datetime_parse_video = []

for col in video_df_model.columns:
    if col == TARGET_VIDEO:
        continue

    if pd.api.types.is_datetime64_any_dtype(video_df_model[col]):
        datetime_cols_video.append(col)
        continue

    col_lower = col.lower()
    looks_temporal = any(token in col_lower for token in ["time", "date", "timestamp", "datetime"])

    if looks_temporal:
        try:
            parsed = pd.to_datetime(video_df_model[col], errors="raise")
            video_df_model[col] = parsed
            datetime_cols_video.append(col)
        except Exception:
            failed_datetime_parse_video.append(col)


# Expand datetime features

for col in datetime_cols_video:
    video_df_model[f"{col}_hour"] = video_df_model[col].dt.hour
    video_df_model[f"{col}_minute"] = video_df_model[col].dt.minute
    video_df_model[f"{col}_second"] = video_df_model[col].dt.second
    video_df_model[f"{col}_dayofweek"] = video_df_model[col].dt.dayofweek
    video_df_model[f"{col}_unix"] = video_df_model[col].astype("int64") // 10**9

if len(datetime_cols_video) > 0:
    video_df_model = video_df_model.drop(columns=datetime_cols_video)


# Column roles

context_cols_video = [c for c in ["camera_id", "zone_id"] if c in video_df_model.columns]
exclude_from_model_video = []

X_video_full = video_df_model.drop(columns=[TARGET_VIDEO]).copy()
X_video_model = video_df_model.drop(columns=[TARGET_VIDEO] + exclude_from_model_video, errors="ignore").copy()
y_video = video_df_model[TARGET_VIDEO].astype(int).copy()

num_cols_video = X_video_model.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_video = X_video_model.select_dtypes(exclude=[np.number]).columns.tolist()


# Cleaning summary

video_cleaning_summary = {
    "target_column": TARGET_VIDEO,
    "rows_before_dedup": rows_before_dedup,
    "rows_after_dedup": rows_after_dedup,
    "duplicates_removed": duplicates_removed,
    "datetime_columns_detected": datetime_cols_video,
    "datetime_parse_failed_columns": failed_datetime_parse_video,
    "context_columns_available": context_cols_video,
    "columns_excluded_from_model": exclude_from_model_video,
    "n_model_features_total": int(X_video_model.shape[1]),
    "n_model_numeric_features": int(len(num_cols_video)),
    "n_model_categorical_features": int(len(cat_cols_video)),
    "target_positive_ratio": float(y_video.mean()),
    "target_distribution": {str(k): int(v) for k, v in y_video.value_counts().sort_index().to_dict().items()}
}

save_json_file(
    video_cleaning_summary,
    "video_cleaning_summary.json",
    category="preprocessing",
    description="Video data cleaning and feature summary"
)

feature_role_video_df = pd.DataFrame({
    "column": X_video_full.columns,
    "used_for_model": [c in X_video_model.columns for c in X_video_full.columns],
    "is_numeric_in_model": [c in num_cols_video for c in X_video_full.columns],
    "is_categorical_in_model": [c in cat_cols_video for c in X_video_full.columns],
    "excluded_from_model": [c in exclude_from_model_video for c in X_video_full.columns]
})

save_df(
    feature_role_video_df,
    "video_feature_role_summary.csv",
    description="Video feature inclusion and exclusion summary"
)

append_summary(f"Video target: {TARGET_VIDEO}")
append_summary(f"Video rows before dedup: {rows_before_dedup}")
append_summary(f"Video rows after dedup: {rows_after_dedup}")
append_summary(f"Video duplicates removed: {duplicates_removed}")
append_summary(f"Video datetime columns handled: {datetime_cols_video}")
append_summary(f"Video datetime parse failed columns: {failed_datetime_parse_video}")
append_summary(f"Video context columns available: {context_cols_video}")
append_summary(f"Video columns excluded from model: {exclude_from_model_video}")
append_summary(f"Video positive ratio: {float(y_video.mean()):.6f}")
append_summary(f"Video model numeric columns: {len(num_cols_video)}")
append_summary(f"Video model categorical columns: {len(cat_cols_video)}")

print("Selected video target column:", TARGET_VIDEO)

print("\nVideo target distribution:")
print(y_video.value_counts(normalize=True).sort_index())

print("\nVideo target counts:")
print(y_video.value_counts().sort_index())

print("\nDatetime columns handled:")
print(datetime_cols_video if len(datetime_cols_video) > 0 else "None")

print("\nColumns excluded from model:")
print(exclude_from_model_video if len(exclude_from_model_video) > 0 else "None")

print("\nModel feature matrix shape:")
print(X_video_model.shape)

print(f"\nNumeric model columns      : {len(num_cols_video)}")
print(f"Categorical model columns  : {len(cat_cols_video)}")
print(f"Active run ID              : {RUN_ID}")

Saved json: outputs\run_20260409_172935_seed42\others\video_cleaning_summary.json
Saved table: outputs\run_20260409_172935_seed42\tables\video_feature_role_summary.csv
Selected video target column: suspicious_label

Video target distribution:
suspicious_label
0    0.180347
1    0.819653
Name: proportion, dtype: float64

Video target counts:
suspicious_label
0     38061
1    172982
Name: count, dtype: int64

Datetime columns handled:
['timestamp']

Columns excluded from model:
None

Model feature matrix shape:
(211043, 10)

Numeric model columns      : 10
Categorical model columns  : 0
Active run ID              : run_20260409_172935_seed42


In [ ]:

# Train-test split


# Confirm active modeling matrices
assert "X_cyber_model" in globals(), "X_cyber_model is missing"
assert "y_cyber" in globals(), "y_cyber is missing"
assert "X_video_model" in globals(), "X_video_model is missing"
assert "y_video" in globals(), "y_video is missing"

Xc = X_cyber_model.copy()
Xv = X_video_model.copy()

print("Source matrices before split:")
print(f"Cyber X shape             : {Xc.shape}")
print(f"Cyber y length            : {len(y_cyber)}")
print(f"Video X shape             : {Xv.shape}")
print(f"Video y length            : {len(y_video)}")
print(f"Active run ID             : {RUN_ID}")

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc,
    y_cyber,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
    stratify=y_cyber if CONFIG.get("stratify_cyber", True) else None
)

Xv_train, Xv_test, yv_train, yv_test = train_test_split(
    Xv,
    y_video,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
    stratify=y_video if CONFIG.get("stratify_video", True) else None
)

split_summary = {
    "active_run_id": RUN_ID,
    "cyber_source_shape": [int(Xc.shape[0]), int(Xc.shape[1])],
    "video_source_shape": [int(Xv.shape[0]), int(Xv.shape[1])],
    "cyber_train_shape": [int(Xc_train.shape[0]), int(Xc_train.shape[1])],
    "cyber_test_shape": [int(Xc_test.shape[0]), int(Xc_test.shape[1])],
    "video_train_shape": [int(Xv_train.shape[0]), int(Xv_train.shape[1])],
    "video_test_shape": [int(Xv_test.shape[0]), int(Xv_test.shape[1])],
    "test_size": float(CONFIG["test_size"]),
    "random_state": int(CONFIG["random_state"]),
    "stratify_cyber": bool(CONFIG.get("stratify_cyber", True)),
    "stratify_video": bool(CONFIG.get("stratify_video", True)),
    "cyber_train_positive_ratio": float(yc_train.mean()),
    "cyber_test_positive_ratio": float(yc_test.mean()),
    "video_train_positive_ratio": float(yv_train.mean()),
    "video_test_positive_ratio": float(yv_test.mean())
}

save_json_file(
    split_summary,
    "train_test_split_summary.json",
    category="split",
    description="Train-test split summary for cyber and video branches"
)

split_table = pd.DataFrame({
    "branch": ["cyber", "cyber", "video", "video"],
    "split": ["train", "test", "train", "test"],
    "n_rows": [len(Xc_train), len(Xc_test), len(Xv_train), len(Xv_test)],
    "n_features": [Xc_train.shape[1], Xc_test.shape[1], Xv_train.shape[1], Xv_test.shape[1]],
    "positive_ratio": [yc_train.mean(), yc_test.mean(), yv_train.mean(), yv_test.mean()]
})

save_df(
    split_table,
    "train_test_split_table.csv",
    description="Train-test split sizes and class ratios"
)

append_summary(f"Cyber source shape: {Xc.shape}")
append_summary(f"Video source shape: {Xv.shape}")
append_summary(f"Cyber train shape: {Xc_train.shape}")
append_summary(f"Cyber test shape: {Xc_test.shape}")
append_summary(f"Video train shape: {Xv_train.shape}")
append_summary(f"Video test shape: {Xv_test.shape}")
append_summary(f"Cyber train positive ratio: {float(yc_train.mean()):.6f}")
append_summary(f"Cyber test positive ratio: {float(yc_test.mean()):.6f}")
append_summary(f"Video train positive ratio: {float(yv_train.mean()):.6f}")
append_summary(f"Video test positive ratio: {float(yv_test.mean()):.6f}")

print("Splits created.")

print("\nCyber split:")
print(f"Train shape                : {Xc_train.shape}")
print(f"Test shape                 : {Xc_test.shape}")
print(f"Train positive ratio       : {yc_train.mean():.6f}")
print(f"Test positive ratio        : {yc_test.mean():.6f}")

print("\nVideo split:")
print(f"Train shape                : {Xv_train.shape}")
print(f"Test shape                 : {Xv_test.shape}")
print(f"Train positive ratio       : {yv_train.mean():.6f}")
print(f"Test positive ratio        : {yv_test.mean():.6f}")

Source matrices before split:
Cyber X shape             : (190474, 40)
Cyber y length            : 190474
Video X shape             : (211043, 10)
Video y length            : 211043
Active run ID             : run_20260409_172935_seed42
Saved json: outputs\run_20260409_172935_seed42\others\train_test_split_summary.json
Saved table: outputs\run_20260409_172935_seed42\tables\train_test_split_table.csv
Splits created.

Cyber split:
Train shape                : (133331, 40)
Test shape                 : (57143, 40)
Train positive ratio       : 0.779286
Test positive ratio        : 0.779291

Video split:
Train shape                : (147730, 10)
Test shape                 : (63313, 10)
Train positive ratio       : 0.819468
Test positive ratio        : 0.820084


In [ ]:

# Preprocessing pipelines


assert "num_cols_cyber" in globals(), "num_cols_cyber is missing"
assert "cat_cols_cyber" in globals(), "cat_cols_cyber is missing"
assert "num_cols_video" in globals(), "num_cols_video is missing"
assert "cat_cols_video" in globals(), "cat_cols_video is missing"

print("Column groups before building preprocessors:")
print(f"Cyber numeric columns      : {len(num_cols_cyber)}")
print(f"Cyber categorical columns  : {len(cat_cols_cyber)}")
print(f"Video numeric columns      : {len(num_cols_video)}")
print(f"Video categorical columns  : {len(cat_cols_video)}")
print(f"Active run ID              : {RUN_ID}")

def make_preprocessor(num_cols, cat_cols, use_scaling=True):
    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]
    if use_scaling:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(steps=numeric_steps)

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    transformers = []
    if len(num_cols) > 0:
        transformers.append(("num", numeric_transformer, num_cols))
    if len(cat_cols) > 0:
        transformers.append(("cat", categorical_transformer, cat_cols))

    return ColumnTransformer(transformers=transformers)

pre_cyber = make_preprocessor(
    num_cols=num_cols_cyber,
    cat_cols=cat_cols_cyber,
    use_scaling=True
)

pre_video = make_preprocessor(
    num_cols=num_cols_video,
    cat_cols=cat_cols_video,
    use_scaling=True
)

preprocessing_summary = {
    "active_run_id": RUN_ID,
    "cyber_numeric_cols": len(num_cols_cyber),
    "cyber_categorical_cols": len(cat_cols_cyber),
    "video_numeric_cols": len(num_cols_video),
    "video_categorical_cols": len(cat_cols_video),
    "numeric_imputer_strategy": "median",
    "categorical_imputer_strategy": "most_frequent",
    "categorical_encoder": "OneHotEncoder(handle_unknown='ignore')",
    "numeric_scaling_enabled": True
}

save_json_file(
    preprocessing_summary,
    "preprocessing_summary.json",
    category="preprocessing",
    description="Preprocessing pipeline summary"
)

append_summary("Preprocessing pipelines defined.")
append_summary(f"Cyber numeric columns: {len(num_cols_cyber)}")
append_summary(f"Cyber categorical columns: {len(cat_cols_cyber)}")
append_summary(f"Video numeric columns: {len(num_cols_video)}")
append_summary(f"Video categorical columns: {len(cat_cols_video)}")

print("Preprocessors ready.")
print(f"Cyber numeric columns      : {len(num_cols_cyber)}")
print(f"Cyber categorical columns  : {len(cat_cols_cyber)}")
print(f"Video numeric columns      : {len(num_cols_video)}")
print(f"Video categorical columns  : {len(cat_cols_video)}")
print(f"Active run ID              : {RUN_ID}")

Column groups before building preprocessors:
Cyber numeric columns      : 16
Cyber categorical columns  : 24
Video numeric columns      : 10
Video categorical columns  : 0
Active run ID              : run_20260409_172935_seed42
Saved json: outputs\run_20260409_172935_seed42\others\preprocessing_summary.json
Preprocessors ready.
Cyber numeric columns      : 16
Cyber categorical columns  : 24
Video numeric columns      : 10
Video categorical columns  : 0
Active run ID              : run_20260409_172935_seed42


In [ ]:

# Cyber model training


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score
)

assert "Xc_train" in globals(), "Xc_train is missing"
assert "Xc_test" in globals(), "Xc_test is missing"
assert "yc_train" in globals(), "yc_train is missing"
assert "yc_test" in globals(), "yc_test is missing"

print("Cyber training source state:")
print(f"Xc_train shape             : {Xc_train.shape}")
print(f"Xc_test shape              : {Xc_test.shape}")
print(f"yc_train length            : {len(yc_train)}")
print(f"yc_test length             : {len(yc_test)}")
print(f"Active run ID              : {RUN_ID}")

# Optional strict exclusion pass
STRICT_EXCLUDE_CYBER = ["type", "src_ip", "dst_ip"]
Xc_train_use = Xc_train.drop(columns=[c for c in STRICT_EXCLUDE_CYBER if c in Xc_train.columns], errors="ignore").copy()
Xc_test_use  = Xc_test.drop(columns=[c for c in STRICT_EXCLUDE_CYBER if c in Xc_test.columns], errors="ignore").copy()

print("\nCyber training columns after strict exclusion:")
print(Xc_train_use.columns.tolist())
print(f"Number of features used    : {Xc_train_use.shape[1]}")

cyber_models = {
    "LogReg": LogisticRegression(max_iter=1000, random_state=SEED),
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1
    )
}

cyber_results = []
cyber_pipes = {}

for model_name, model in cyber_models.items():
    print("=" * 80)
    print(f"Training cyber model: {model_name}")

    try:
        pipe = Pipeline(steps=[
            ("preprocessor", make_preprocessor(
                num_cols=Xc_train_use.select_dtypes(include=[np.number]).columns.tolist(),
                cat_cols=Xc_train_use.select_dtypes(exclude=[np.number]).columns.tolist(),
                use_scaling=True
            )),
            ("model", model)
        ])

        start_train = time.time()
        pipe.fit(Xc_train_use, yc_train)
        train_elapsed = time.time() - start_train

        start_pred = time.time()
        yc_pred = pipe.predict(Xc_test_use)
        pred_elapsed = time.time() - start_pred

        yc_prob = None
        if hasattr(pipe.named_steps["model"], "predict_proba"):
            yc_prob = pipe.predict_proba(Xc_test_use)[:, 1]

        tn, fp, fn, tp = confusion_matrix(yc_test, yc_pred).ravel()

        metrics = {
            "model": model_name,
            "n_features": int(Xc_train_use.shape[1]),
            "accuracy": accuracy_score(yc_test, yc_pred),
            "balanced_accuracy": balanced_accuracy_score(yc_test, yc_pred),
            "precision": precision_score(yc_test, yc_pred, zero_division=0),
            "recall": recall_score(yc_test, yc_pred, zero_division=0),
            "f1": f1_score(yc_test, yc_pred, zero_division=0),
            "roc_auc": roc_auc_score(yc_test, yc_prob) if yc_prob is not None else np.nan,
            "pr_auc": average_precision_score(yc_test, yc_prob) if yc_prob is not None else np.nan,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
            "fpr": float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
            "train_time_sec": round(train_elapsed, 4),
            "predict_time_sec": round(pred_elapsed, 4),
            "test_rows": int(len(Xc_test_use))
        }

        cyber_results.append(metrics)
        cyber_pipes[model_name] = pipe

        log_metric_block(f"Cyber model - {model_name}", metrics)
        save_metrics_json(f"Cyber model - {model_name}", metrics, f"cyber_metrics_{model_name}.json")

        print("Done:", metrics)

    except Exception as e:
        print(f"Failed for {model_name}: {e}")
        append_summary(f"Cyber model failed - {model_name}: {e}")

if len(cyber_results) == 0:
    raise RuntimeError("All cyber models failed.")

cyber_results_df = pd.DataFrame(cyber_results).sort_values(
    by=["f1", "pr_auc", "roc_auc"],
    ascending=False
).reset_index(drop=True)

save_df(
    cyber_results_df,
    "cyber_model_results.csv",
    description="Cyber model comparison results"
)

append_summary("Cyber model training completed successfully.")
print(cyber_results_df)

Cyber training source state:
Xc_train shape             : (133331, 40)
Xc_test shape              : (57143, 40)
yc_train length            : 133331
yc_test length             : 57143
Active run ID              : run_20260409_172935_seed42

Cyber training columns after strict exclusion:
['src_port', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice']
Number of features used    : 40
Training cyber model: LogReg

----------------------------------------------

In [ ]:

# Video model training with strict feature exclusion


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score
)

assert "Xv_train" in globals(), "Xv_train is missing"
assert "Xv_test" in globals(), "Xv_test is missing"
assert "yv_train" in globals(), "yv_train is missing"
assert "yv_test" in globals(), "yv_test is missing"

print("Video training source state:")
print(f"Xv_train shape             : {Xv_train.shape}")
print(f"Xv_test shape              : {Xv_test.shape}")
print(f"yv_train length            : {len(yv_train)}")
print(f"yv_test length             : {len(yv_test)}")
print(f"Active run ID              : {RUN_ID}")

STRICT_EXCLUDE_VIDEO = [
    "authenticity_score",
    "temporal_consistency",
    "uncertainty_score"
]

Xv_train_use = Xv_train.drop(columns=[c for c in STRICT_EXCLUDE_VIDEO if c in Xv_train.columns], errors="ignore").copy()
Xv_test_use  = Xv_test.drop(columns=[c for c in STRICT_EXCLUDE_VIDEO if c in Xv_test.columns], errors="ignore").copy()

print("\nVideo training columns after strict exclusion:")
print(Xv_train_use.columns.tolist())
print(f"Number of features used    : {Xv_train_use.shape[1]}")

video_models = {
    "LogReg": LogisticRegression(max_iter=1000, random_state=SEED),
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1
    )
}

video_results = []
video_pipes = {}

for model_name, model in video_models.items():
    print("=" * 80)
    print(f"Training video model: {model_name}")

    try:
        pipe = Pipeline(steps=[
            ("preprocessor", make_preprocessor(
                num_cols=Xv_train_use.select_dtypes(include=[np.number]).columns.tolist(),
                cat_cols=Xv_train_use.select_dtypes(exclude=[np.number]).columns.tolist(),
                use_scaling=True
            )),
            ("model", model)
        ])

        start_train = time.time()
        pipe.fit(Xv_train_use, yv_train)
        train_elapsed = time.time() - start_train

        start_pred = time.time()
        yv_pred = pipe.predict(Xv_test_use)
        pred_elapsed = time.time() - start_pred

        yv_prob = None
        if hasattr(pipe.named_steps["model"], "predict_proba"):
            yv_prob = pipe.predict_proba(Xv_test_use)[:, 1]

        tn, fp, fn, tp = confusion_matrix(yv_test, yv_pred).ravel()

        metrics = {
            "model": model_name,
            "n_features": int(Xv_train_use.shape[1]),
            "accuracy": accuracy_score(yv_test, yv_pred),
            "balanced_accuracy": balanced_accuracy_score(yv_test, yv_pred),
            "precision": precision_score(yv_test, yv_pred, zero_division=0),
            "recall": recall_score(yv_test, yv_pred, zero_division=0),
            "f1": f1_score(yv_test, yv_pred, zero_division=0),
            "roc_auc": roc_auc_score(yv_test, yv_prob) if yv_prob is not None else np.nan,
            "pr_auc": average_precision_score(yv_test, yv_prob) if yv_prob is not None else np.nan,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
            "fpr": float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
            "train_time_sec": round(train_elapsed, 4),
            "predict_time_sec": round(pred_elapsed, 4),
            "test_rows": int(len(Xv_test_use))
        }

        video_results.append(metrics)
        video_pipes[model_name] = pipe

        log_metric_block(f"Video model - {model_name}", metrics)
        save_metrics_json(f"Video model - {model_name}", metrics, f"video_metrics_{model_name}.json")

        print("Done:", metrics)

    except Exception as e:
        print(f"Failed for {model_name}: {e}")
        append_summary(f"Video model failed - {model_name}: {e}")

if len(video_results) == 0:
    raise RuntimeError("All video models failed.")

video_results_df = pd.DataFrame(video_results).sort_values(
    by=["balanced_accuracy", "roc_auc", "f1"],
    ascending=False
).reset_index(drop=True)

save_df(
    video_results_df,
    "video_model_results_strict.csv",
    description="Video model comparison with strict exclusion of target-generation features"
)

append_summary("Strict video model training completed successfully.")
print(video_results_df)

Video training source state:
Xv_train shape             : (147730, 10)
Xv_test shape              : (63313, 10)
yv_train length            : 147730
yv_test length             : 63313
Active run ID              : run_20260409_172935_seed42

Video training columns after strict exclusion:
['camera_id', 'zone_id', 'timestamp_hour', 'timestamp_minute', 'timestamp_second', 'timestamp_dayofweek', 'timestamp_unix']
Number of features used    : 7
Training video model: LogReg

--------------------------------------------------------------------------------
Video model - LogReg
--------------------------------------------------------------------------------
model: LogReg
n_features: 7
accuracy: 0.820084
balanced_accuracy: 0.5
precision: 0.820084
recall: 1.0
f1: 0.90115
roc_auc: 0.497581
pr_auc: 0.818178
tn: 0
fp: 11391
fn: 0
tp: 51922
fpr: 1.0
train_time_sec: 0.1718
predict_time_sec: 0.0096
test_rows: 63313
Saved json: outputs\run_20260409_172935_seed42\others\video_metrics_LogReg.json (overwritt

In [ ]:

# Video model training (full feature set)


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score
)

assert "Xv_train" in globals(), "Xv_train is missing"
assert "Xv_test" in globals(), "Xv_test is missing"
assert "yv_train" in globals(), "yv_train is missing"
assert "yv_test" in globals(), "yv_test is missing"

print("Video training source state (full feature set):")
print(f"Xv_train shape             : {Xv_train.shape}")
print(f"Xv_test shape              : {Xv_test.shape}")
print(f"yv_train length            : {len(yv_train)}")
print(f"yv_test length             : {len(yv_test)}")
print(f"Active run ID              : {RUN_ID}")

Xv_train_full = Xv_train.copy()
Xv_test_full  = Xv_test.copy()

print("\nVideo training columns (full feature set):")
print(Xv_train_full.columns.tolist())
print(f"Number of features used    : {Xv_train_full.shape[1]}")

video_models_full = {
    "LogReg": LogisticRegression(max_iter=1000, random_state=SEED),
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1
    )
}

video_results_full = []
video_pipes_full = {}

for model_name, model in video_models_full.items():
    print("=" * 80)
    print(f"Training full video model: {model_name}")

    try:
        pipe = Pipeline(steps=[
            ("preprocessor", make_preprocessor(
                num_cols=Xv_train_full.select_dtypes(include=[np.number]).columns.tolist(),
                cat_cols=Xv_train_full.select_dtypes(exclude=[np.number]).columns.tolist(),
                use_scaling=True
            )),
            ("model", model)
        ])

        start_train = time.time()
        pipe.fit(Xv_train_full, yv_train)
        train_elapsed = time.time() - start_train

        start_pred = time.time()
        yv_pred = pipe.predict(Xv_test_full)
        pred_elapsed = time.time() - start_pred

        yv_prob = None
        if hasattr(pipe.named_steps["model"], "predict_proba"):
            yv_prob = pipe.predict_proba(Xv_test_full)[:, 1]

        tn, fp, fn, tp = confusion_matrix(yv_test, yv_pred).ravel()

        metrics = {
            "model": model_name,
            "n_features": int(Xv_train_full.shape[1]),
            "accuracy": accuracy_score(yv_test, yv_pred),
            "balanced_accuracy": balanced_accuracy_score(yv_test, yv_pred),
            "precision": precision_score(yv_test, yv_pred, zero_division=0),
            "recall": recall_score(yv_test, yv_pred, zero_division=0),
            "f1": f1_score(yv_test, yv_pred, zero_division=0),
            "roc_auc": roc_auc_score(yv_test, yv_prob) if yv_prob is not None else np.nan,
            "pr_auc": average_precision_score(yv_test, yv_prob) if yv_prob is not None else np.nan,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
            "fpr": float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
            "train_time_sec": round(train_elapsed, 4),
            "predict_time_sec": round(pred_elapsed, 4),
            "test_rows": int(len(Xv_test_full))
        }

        video_results_full.append(metrics)
        video_pipes_full[model_name] = pipe

        log_metric_block(f"Video full model - {model_name}", metrics)
        save_metrics_json(f"Video full model - {model_name}", metrics, f"video_full_metrics_{model_name}.json")

        print("Done:", metrics)

    except Exception as e:
        print(f"Failed for {model_name}: {e}")
        append_summary(f"Video full model failed - {model_name}: {e}")

if len(video_results_full) == 0:
    raise RuntimeError("All full video models failed.")

video_results_full_df = pd.DataFrame(video_results_full).sort_values(
    by=["balanced_accuracy", "roc_auc", "pr_auc", "f1"],
    ascending=False
).reset_index(drop=True)

save_df(
    video_results_full_df,
    "video_model_results_full.csv",
    description="Video model comparison with full feature set"
)

append_summary("Full video model training completed successfully.")
print(video_results_full_df)

Video training source state (full feature set):
Xv_train shape             : (147730, 10)
Xv_test shape              : (63313, 10)
yv_train length            : 147730
yv_test length             : 63313
Active run ID              : run_20260409_172935_seed42

Video training columns (full feature set):
['camera_id', 'zone_id', 'authenticity_score', 'temporal_consistency', 'uncertainty_score', 'timestamp_hour', 'timestamp_minute', 'timestamp_second', 'timestamp_dayofweek', 'timestamp_unix']
Number of features used    : 10
Training full video model: LogReg

--------------------------------------------------------------------------------
Video full model - LogReg
--------------------------------------------------------------------------------
model: LogReg
n_features: 10
accuracy: 0.970606
balanced_accuracy: 0.936987
precision: 0.975025
recall: 0.989503
f1: 0.982211
roc_auc: 0.991295
pr_auc: 0.997997
tn: 10075
fp: 1316
fn: 545
tp: 51377
fpr: 0.11553
train_time_sec: 0.3162
predict_time_sec: 

In [ ]:

# Figure: Video branch strict-vs-full comparison


assert "video_results_df" in globals(), "video_results_df (strict results) is missing"
assert "video_results_full_df" in globals(), "video_results_full_df (full results) is missing"

strict_df = video_results_df.copy()
strict_df["setting"] = "strict_exclusion"

full_df = video_results_full_df.copy()
full_df["setting"] = "full_feature_set"

plot_df = pd.concat([strict_df, full_df], ignore_index=True)

metrics_to_plot = [
    "balanced_accuracy",
    "roc_auc",
    "pr_auc",
    "fpr"
]

plot_records = []
for _, row in plot_df.iterrows():
    for metric in metrics_to_plot:
        plot_records.append({
            "model": row["model"],
            "setting": row["setting"],
            "metric": metric,
            "value": row[metric]
        })

plot_long_df = pd.DataFrame(plot_records)

save_df(
    plot_long_df,
    "video_strict_vs_full_plot_data.csv",
    description="Long-format data for strict-vs-full video branch comparison plot"
)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, metric in zip(axes, metrics_to_plot):
    sub = plot_long_df[plot_long_df["metric"] == metric].copy()

    models = ["LogReg", "RandomForest", "ExtraTrees"]
    x = np.arange(len(models))
    width = 0.35

    strict_vals = [
        sub[(sub["model"] == m) & (sub["setting"] == "strict_exclusion")]["value"].values[0]
        for m in models
    ]
    full_vals = [
        sub[(sub["model"] == m) & (sub["setting"] == "full_feature_set")]["value"].values[0]
        for m in models
    ]

    ax.bar(x - width/2, strict_vals, width, label="Strict exclusion")
    ax.bar(x + width/2, full_vals, width, label="Full feature set")

    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=0)
    ax.set_title(metric.replace("_", " ").title())
    ax.set_ylabel("Value")
    ax.grid(axis="y", alpha=0.3)

    if metric != "fpr":
        ax.set_ylim(0, 1.05)
    else:
        ax.set_ylim(0, max(max(strict_vals), max(full_vals)) * 1.1 + 1e-6)

axes[0].legend()
fig.suptitle("Video Branch Performance: Strict Exclusion vs Full Feature Set", fontsize=14)

save_figure(
    fig,
    "video_strict_vs_full_comparison.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description="Comparison of strict-exclusion and full-feature video branch performance"
)

append_summary("Saved figure: video_strict_vs_full_comparison.png")

print("Saved video strict-vs-full comparison figure.")
print(plot_long_df.head(12))

Saved table: outputs\run_20260409_172935_seed42\tables\video_strict_vs_full_plot_data.csv
Saved figure: outputs\run_20260409_172935_seed42\figures\video_strict_vs_full_comparison.png
Saved video strict-vs-full comparison figure.
           model           setting             metric     value
0   RandomForest  strict_exclusion  balanced_accuracy  0.500824
1   RandomForest  strict_exclusion            roc_auc  0.499979
2   RandomForest  strict_exclusion             pr_auc  0.821053
3   RandomForest  strict_exclusion                fpr  0.985603
4     ExtraTrees  strict_exclusion  balanced_accuracy  0.500551
5     ExtraTrees  strict_exclusion            roc_auc  0.498629
6     ExtraTrees  strict_exclusion             pr_auc  0.820988
7     ExtraTrees  strict_exclusion                fpr  0.968659
8         LogReg  strict_exclusion  balanced_accuracy  0.500000
9         LogReg  strict_exclusion            roc_auc  0.497581
10        LogReg  strict_exclusion             pr_auc  0.818178
11 

In [ ]:

# Figure: Cyber model comparison


assert "cyber_results_df" in globals(), "cyber_results_df is missing"

cyber_plot_df = cyber_results_df.copy()

metrics_to_plot = [
    "balanced_accuracy",
    "roc_auc",
    "pr_auc",
    "fpr"
]

plot_records = []
for _, row in cyber_plot_df.iterrows():
    for metric in metrics_to_plot:
        plot_records.append({
            "model": row["model"],
            "metric": metric,
            "value": row[metric]
        })

cyber_plot_long_df = pd.DataFrame(plot_records)

save_df(
    cyber_plot_long_df,
    "cyber_model_plot_data.csv",
    description="Long-format data for cyber model comparison plot"
)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

models = ["LogReg", "RandomForest", "ExtraTrees"]

for ax, metric in zip(axes, metrics_to_plot):
    sub = cyber_plot_long_df[cyber_plot_long_df["metric"] == metric].copy()
    vals = [sub[sub["model"] == m]["value"].values[0] for m in models]

    x = np.arange(len(models))
    ax.bar(x, vals)
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=0)
    ax.set_title(metric.replace("_", " ").title())
    ax.set_ylabel("Value")
    ax.grid(axis="y", alpha=0.3)

    if metric != "fpr":
        ax.set_ylim(0, 1.05)
    else:
        ax.set_ylim(0, max(vals) * 1.15 + 1e-6)

fig.suptitle("Cyber Branch Model Comparison", fontsize=14)

save_figure(
    fig,
    "cyber_model_comparison.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description="Comparison of cyber branch model performance"
)

append_summary("Saved figure: cyber_model_comparison.png")

print("Saved cyber model comparison figure.")
print(cyber_plot_long_df.head(12))

Saved table: outputs\run_20260409_172935_seed42\tables\cyber_model_plot_data.csv
Saved figure: outputs\run_20260409_172935_seed42\figures\cyber_model_comparison.png
Saved cyber model comparison figure.
           model             metric     value
0     ExtraTrees  balanced_accuracy  0.998087
1     ExtraTrees            roc_auc  0.999969
2     ExtraTrees             pr_auc  0.999987
3     ExtraTrees                fpr  0.003489
4   RandomForest  balanced_accuracy  0.998008
5   RandomForest            roc_auc  0.999989
6   RandomForest             pr_auc  0.999997
7   RandomForest                fpr  0.003647
8         LogReg  balanced_accuracy  0.984263
9         LogReg            roc_auc  0.996757
10        LogReg             pr_auc  0.998804
11        LogReg                fpr  0.023232


In [ ]:

# Figure: Confusion matrices for selected branch models


from sklearn.metrics import ConfusionMatrixDisplay

assert "best_cyber_name" in globals() or "cyber_results_df" in globals()
assert "cyber_pipes" in globals(), "cyber_pipes is missing"
assert "video_pipes_full" in globals(), "video_pipes_full is missing"

# Select best cyber model from approved cyber results
cyber_rank_df = cyber_results_df.sort_values(
    by=["fpr", "balanced_accuracy", "pr_auc", "roc_auc"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

best_cyber_name = cyber_rank_df.iloc[0]["model"]
best_cyber_pipe = cyber_pipes[best_cyber_name]

# Select best full-feature video model
video_rank_full_df = video_results_full_df.sort_values(
    by=["balanced_accuracy", "roc_auc", "fpr", "pr_auc"],
    ascending=[False, False, True, False]
).reset_index(drop=True)

best_video_name = video_rank_full_df.iloc[0]["model"]
best_video_pipe = video_pipes_full[best_video_name]

# Predictions
yc_pred_best = best_cyber_pipe.predict(Xc_test)
yv_pred_best = best_video_pipe.predict(Xv_test)

cm_cyber = confusion_matrix(yc_test, yc_pred_best)
cm_video = confusion_matrix(yv_test, yv_pred_best)

cm_summary_df = pd.DataFrame({
    "branch": ["cyber", "video"],
    "selected_model": [best_cyber_name, best_video_name],
    "tn": [int(cm_cyber[0, 0]), int(cm_video[0, 0])],
    "fp": [int(cm_cyber[0, 1]), int(cm_video[0, 1])],
    "fn": [int(cm_cyber[1, 0]), int(cm_video[1, 0])],
    "tp": [int(cm_cyber[1, 1]), int(cm_video[1, 1])]
})

save_df(
    cm_summary_df,
    "selected_model_confusion_matrices.csv",
    description="Confusion matrix counts for selected cyber and video models"
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

ConfusionMatrixDisplay(cm_cyber, display_labels=[0, 1]).plot(ax=axes[0], colorbar=False)
axes[0].set_title(f"Cyber: {best_cyber_name}")

ConfusionMatrixDisplay(cm_video, display_labels=[0, 1]).plot(ax=axes[1], colorbar=False)
axes[1].set_title(f"Video: {best_video_name}")

fig.suptitle("Confusion Matrices for Selected Branch Models", fontsize=14)

save_figure(
    fig,
    "selected_branch_confusion_matrices.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description="Confusion matrices for selected cyber and video branch models"
)

append_summary(f"Selected cyber model for branch reporting: {best_cyber_name}")
append_summary(f"Selected video model for branch reporting: {best_video_name}")
append_summary("Saved figure: selected_branch_confusion_matrices.png")

print("Selected cyber model:", best_cyber_name)
print("Selected video model:", best_video_name)
print(cm_summary_df)

Saved table: outputs\run_20260409_172935_seed42\tables\selected_model_confusion_matrices.csv
Saved figure: outputs\run_20260409_172935_seed42\figures\selected_branch_confusion_matrices.png
Selected cyber model: ExtraTrees
Selected video model: RandomForest
  branch selected_model     tn  fp  fn     tp
0  cyber     ExtraTrees  12568  44  15  44516
1  video   RandomForest  11391   0   1  51921


In [ ]:

# Figures: ROC and PR curves for selected branch models


from sklearn.metrics import roc_curve, precision_recall_curve

assert "best_cyber_pipe" in globals(), "best_cyber_pipe is missing"
assert "best_video_pipe" in globals(), "best_video_pipe is missing"

# Probabilities
yc_prob_best = best_cyber_pipe.predict_proba(Xc_test)[:, 1]
yv_prob_best = best_video_pipe.predict_proba(Xv_test)[:, 1]

# ROC
fpr_c, tpr_c, _ = roc_curve(yc_test, yc_prob_best)
fpr_v, tpr_v, _ = roc_curve(yv_test, yv_prob_best)

# PR
prec_c, rec_c, _ = precision_recall_curve(yc_test, yc_prob_best)
prec_v, rec_v, _ = precision_recall_curve(yv_test, yv_prob_best)

curve_summary_df = pd.DataFrame({
    "branch": ["cyber", "video"],
    "selected_model": [best_cyber_name, best_video_name],
    "roc_auc": [
        roc_auc_score(yc_test, yc_prob_best),
        roc_auc_score(yv_test, yv_prob_best)
    ],
    "pr_auc": [
        average_precision_score(yc_test, yc_prob_best),
        average_precision_score(yv_test, yv_prob_best)
    ]
})

save_df(
    curve_summary_df,
    "selected_model_curve_summary.csv",
    description="ROC-AUC and PR-AUC for selected cyber and video models"
)


# ROC figure

fig_roc = plt.figure(figsize=(6.5, 5))
ax = fig_roc.add_subplot(111)

ax.plot(fpr_c, tpr_c, label=f"Cyber ({best_cyber_name})")
ax.plot(fpr_v, tpr_v, label=f"Video ({best_video_name})")
ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves for Selected Branch Models")
ax.legend()
ax.grid(alpha=0.3)

save_figure(
    fig_roc,
    "selected_branch_roc_curves.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description="ROC curves for selected cyber and video branch models"
)


# PR figure

fig_pr = plt.figure(figsize=(6.5, 5))
ax = fig_pr.add_subplot(111)

ax.plot(rec_c, prec_c, label=f"Cyber ({best_cyber_name})")
ax.plot(rec_v, prec_v, label=f"Video ({best_video_name})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves for Selected Branch Models")
ax.legend()
ax.grid(alpha=0.3)

save_figure(
    fig_pr,
    "selected_branch_pr_curves.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description="Precision-recall curves for selected cyber and video branch models"
)

append_summary("Saved figure: selected_branch_roc_curves.png")
append_summary("Saved figure: selected_branch_pr_curves.png")

print("Saved ROC and PR figures for selected branch models.")
print(curve_summary_df)

Saved table: outputs\run_20260409_172935_seed42\tables\selected_model_curve_summary.csv
Saved figure: outputs\run_20260409_172935_seed42\figures\selected_branch_roc_curves.png
Saved figure: outputs\run_20260409_172935_seed42\figures\selected_branch_pr_curves.png
Saved ROC and PR figures for selected branch models.
  branch selected_model   roc_auc    pr_auc
0  cyber     ExtraTrees  0.999969  0.999987
1  video   RandomForest  1.000000  1.000000


In [ ]:

# Select branch models for fusion


assert "cyber_results_df" in globals(), "cyber_results_df is missing"
assert "cyber_pipes" in globals(), "cyber_pipes is missing"
assert "video_results_full_df" in globals(), "video_results_full_df is missing"
assert "video_pipes_full" in globals(), "video_pipes_full is missing"

# Cyber: prefer stronger discrimination with lower false positive rate
cyber_rank_df = cyber_results_df.sort_values(
    by=["fpr", "balanced_accuracy", "pr_auc", "roc_auc"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

best_cyber_name = cyber_rank_df.iloc[0]["model"]
best_cyber_pipe = cyber_pipes[best_cyber_name]

# Video: use full-feature operational results, not strict ablation
video_rank_full_df = video_results_full_df.sort_values(
    by=["balanced_accuracy", "roc_auc", "fpr", "pr_auc"],
    ascending=[False, False, True, False]
).reset_index(drop=True)

best_video_name = video_rank_full_df.iloc[0]["model"]
best_video_pipe = video_pipes_full[best_video_name]

fusion_model_selection_df = pd.DataFrame({
    "branch": ["cyber", "video"],
    "selected_model": [best_cyber_name, best_video_name],
    "selection_basis": [
        "minimize FPR, then maximize balanced_accuracy, PR-AUC, ROC-AUC",
        "maximize balanced_accuracy and ROC-AUC, then minimize FPR and maximize PR-AUC"
    ]
})

save_df(
    fusion_model_selection_df,
    "fusion_model_selection.csv",
    description="Selected operational cyber and video models for fusion"
)

append_summary(f"Selected cyber model for fusion: {best_cyber_name}")
append_summary(f"Selected video model for fusion: {best_video_name}")

print("Selected cyber model for fusion:", best_cyber_name)
print("Selected video model for fusion:", best_video_name)
print(fusion_model_selection_df)

Saved table: outputs\run_20260409_172935_seed42\tables\fusion_model_selection.csv
Selected cyber model for fusion: ExtraTrees
Selected video model for fusion: RandomForest
  branch selected_model                                    selection_basis
0  cyber     ExtraTrees  minimize FPR, then maximize balanced_accuracy,...
1  video   RandomForest  maximize balanced_accuracy and ROC-AUC, then m...


In [ ]:

# Build fusion table from selected branch models


assert "best_cyber_pipe" in globals(), "best_cyber_pipe is missing"
assert "best_video_pipe" in globals(), "best_video_pipe is missing"
assert "Xc_test" in globals() and "yc_test" in globals(), "Cyber test split missing"
assert "Xv_test" in globals() and "yv_test" in globals(), "Video test split missing"


# Branch probabilities

cyber_prob_test = best_cyber_pipe.predict_proba(Xc_test)[:, 1]
video_prob_test = best_video_pipe.predict_proba(Xv_test)[:, 1]

cyber_pred_test = (cyber_prob_test >= CONFIG["cyber_alert_threshold"]).astype(int)
video_pred_test = (video_prob_test >= CONFIG["video_alert_threshold"]).astype(int)


# Align branch outputs
# Current notebook uses independently split branches, so align by
# common minimum test length for fusion-stage experimentation.

n_fusion = min(len(Xc_test), len(Xv_test))

fusion_df = pd.DataFrame({
    "fusion_index": np.arange(n_fusion, dtype=np.int64),

    "cyber_prob": cyber_prob_test[:n_fusion],
    "cyber_pred": cyber_pred_test[:n_fusion],
    "cyber_label": yc_test.iloc[:n_fusion].to_numpy() if hasattr(yc_test, "iloc") else np.asarray(yc_test)[:n_fusion],

    "video_prob": video_prob_test[:n_fusion],
    "video_pred": video_pred_test[:n_fusion],
    "video_label": yv_test.iloc[:n_fusion].to_numpy() if hasattr(yv_test, "iloc") else np.asarray(yv_test)[:n_fusion]
})


# Optional uncertainty proxies
# For binary probabilities, smaller margin to 0.5 means higher uncertainty.

fusion_df["cyber_uncertainty"] = 1.0 - np.abs(fusion_df["cyber_prob"] - 0.5) * 2.0
fusion_df["video_uncertainty"] = 1.0 - np.abs(fusion_df["video_prob"] - 0.5) * 2.0


# Fusion risk score
# weighted_mean baseline; equal weights for now

fusion_df["fusion_risk"] = 0.5 * fusion_df["cyber_prob"] + 0.5 * fusion_df["video_prob"]
fusion_df["fusion_pred"] = (fusion_df["fusion_risk"] >= CONFIG["fusion_risk_threshold"]).astype(int)

# A simple composite target for fusion-stage evaluation:
# mark event positive if either modality is labeled suspicious/malicious
fusion_df["fusion_label_or"] = (
    (fusion_df["cyber_label"].astype(int) == 1) |
    (fusion_df["video_label"].astype(int) == 1)
).astype(int)

fusion_df["fusion_label_and"] = (
    (fusion_df["cyber_label"].astype(int) == 1) &
    (fusion_df["video_label"].astype(int) == 1)
).astype(int)


# Save outputs

save_df(
    fusion_df,
    "fusion_test_table.csv",
    description="Aligned fusion-stage test table with cyber/video probabilities, predictions, and fused risk"
)

fusion_build_summary = {
    "active_run_id": RUN_ID,
    "selected_cyber_model": best_cyber_name,
    "selected_video_model": best_video_name,
    "cyber_test_rows": int(len(Xc_test)),
    "video_test_rows": int(len(Xv_test)),
    "fusion_rows_used": int(n_fusion),
    "fusion_strategy": CONFIG["fusion_strategy"],
    "cyber_alert_threshold": float(CONFIG["cyber_alert_threshold"]),
    "video_alert_threshold": float(CONFIG["video_alert_threshold"]),
    "fusion_risk_threshold": float(CONFIG["fusion_risk_threshold"]),
    "fusion_label_definition_primary": "OR of cyber_label and video_label",
    "fusion_label_definition_secondary": "AND of cyber_label and video_label"
}

save_json_file(
    fusion_build_summary,
    "fusion_build_summary.json",
    category="fusion",
    description="Summary of fusion-table construction and selected branch models"
)

append_summary(f"Fusion table built with {n_fusion} rows.")
append_summary(f"Selected cyber model for fusion: {best_cyber_name}")
append_summary(f"Selected video model for fusion: {best_video_name}")
append_summary(f"Fusion strategy: {CONFIG['fusion_strategy']}")

print("Fusion table built successfully.")
print(f"Selected cyber model       : {best_cyber_name}")
print(f"Selected video model       : {best_video_name}")
print(f"Cyber test rows            : {len(Xc_test)}")
print(f"Video test rows            : {len(Xv_test)}")
print(f"Fusion rows used           : {n_fusion}")
print(f"Fusion table shape         : {fusion_df.shape}")

print("\nFusion table preview:")
print(fusion_df.head())

Saved table: outputs\run_20260409_172935_seed42\tables\fusion_test_table.csv
Saved json: outputs\run_20260409_172935_seed42\others\fusion_build_summary.json
Fusion table built successfully.
Selected cyber model       : ExtraTrees
Selected video model       : RandomForest
Cyber test rows            : 57143
Video test rows            : 63313
Fusion rows used           : 57143
Fusion table shape         : (57143, 13)

Fusion table preview:
   fusion_index  cyber_prob  cyber_pred  cyber_label  video_prob  video_pred  \
0             0         1.0           1            1         1.0           1   
1             1         1.0           1            1         1.0           1   
2             2         0.0           0            0         1.0           1   
3             3         1.0           1            1         1.0           1   
4             4         1.0           1            1         1.0           1   

   video_label  cyber_uncertainty  video_uncertainty  fusion_risk  \
0        

In [ ]:

# Fusion-level evaluation


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score
)

assert "fusion_df" in globals(), "fusion_df is missing"

# Primary evaluation target
y_fusion = fusion_df["fusion_label_or"].astype(int).to_numpy()
y_fusion_pred = fusion_df["fusion_pred"].astype(int).to_numpy()
y_fusion_score = fusion_df["fusion_risk"].astype(float).to_numpy()

tn, fp, fn, tp = confusion_matrix(y_fusion, y_fusion_pred).ravel()

fusion_metrics = {
    "selected_cyber_model": best_cyber_name,
    "selected_video_model": best_video_name,
    "fusion_strategy": CONFIG["fusion_strategy"],
    "fusion_rows": int(len(fusion_df)),
    "fusion_accuracy": accuracy_score(y_fusion, y_fusion_pred),
    "fusion_balanced_accuracy": balanced_accuracy_score(y_fusion, y_fusion_pred),
    "fusion_precision": precision_score(y_fusion, y_fusion_pred, zero_division=0),
    "fusion_recall": recall_score(y_fusion, y_fusion_pred, zero_division=0),
    "fusion_f1": f1_score(y_fusion, y_fusion_pred, zero_division=0),
    "fusion_roc_auc": roc_auc_score(y_fusion, y_fusion_score),
    "fusion_pr_auc": average_precision_score(y_fusion, y_fusion_score),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
    "fusion_fpr": float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
    "positive_rate_label_or": float(y_fusion.mean()),
    "positive_rate_pred": float(y_fusion_pred.mean())
}

save_metrics_json(
    "Fusion metrics (OR label definition)",
    fusion_metrics,
    "fusion_metrics_or.json"
)

fusion_metrics_df = pd.DataFrame([fusion_metrics])
save_df(
    fusion_metrics_df,
    "fusion_metrics_or.csv",
    description="Fusion-level evaluation metrics using OR label definition"
)

append_summary("Fusion-level evaluation completed.")
append_summary(f"Fusion rows: {len(fusion_df)}")
append_summary(f"Fusion balanced accuracy: {fusion_metrics['fusion_balanced_accuracy']:.6f}")
append_summary(f"Fusion ROC-AUC: {fusion_metrics['fusion_roc_auc']:.6f}")
append_summary(f"Fusion PR-AUC: {fusion_metrics['fusion_pr_auc']:.6f}")
append_summary(f"Fusion FPR: {fusion_metrics['fusion_fpr']:.6f}")

print("Fusion metrics computed successfully.")
for k, v in fusion_metrics.items():
    print(f"{k}: {v}")

Saved json: outputs\run_20260409_172935_seed42\others\fusion_metrics_or.json
Saved table: outputs\run_20260409_172935_seed42\tables\fusion_metrics_or.csv
Fusion metrics computed successfully.
selected_cyber_model: ExtraTrees
selected_video_model: RandomForest
fusion_strategy: weighted_mean
fusion_rows: 57143
fusion_accuracy: 0.680800797998005
fusion_balanced_accuracy: 0.8336646665085994
fusion_precision: 1.0
fusion_recall: 0.6673293330171989
fusion_f1: 0.8004769301450481
fusion_roc_auc: 0.9999996295546238
fusion_pr_auc: 0.9999999843665268
tn: 2314
fp: 0
fn: 18240
tp: 36589
fusion_fpr: 0.0
positive_rate_label_or: 0.9595051012372469
positive_rate_pred: 0.6403058992352519


In [ ]:

# Figures: Fusion confusion matrix and fused-risk distributions


from sklearn.metrics import ConfusionMatrixDisplay

assert "fusion_df" in globals(), "fusion_df is missing"
assert "fusion_metrics" in globals(), "fusion_metrics is missing"

y_fusion = fusion_df["fusion_label_or"].astype(int).to_numpy()
y_fusion_pred = fusion_df["fusion_pred"].astype(int).to_numpy()
y_fusion_score = fusion_df["fusion_risk"].astype(float).to_numpy()

cm_fusion = confusion_matrix(y_fusion, y_fusion_pred)

# Save detailed summary
fusion_distribution_summary_df = pd.DataFrame({
    "group": ["label_0", "label_1"],
    "count": [
        int((y_fusion == 0).sum()),
        int((y_fusion == 1).sum())
    ],
    "mean_fusion_risk": [
        float(fusion_df.loc[fusion_df["fusion_label_or"] == 0, "fusion_risk"].mean()),
        float(fusion_df.loc[fusion_df["fusion_label_or"] == 1, "fusion_risk"].mean())
    ],
    "median_fusion_risk": [
        float(fusion_df.loc[fusion_df["fusion_label_or"] == 0, "fusion_risk"].median()),
        float(fusion_df.loc[fusion_df["fusion_label_or"] == 1, "fusion_risk"].median())
    ]
})

save_df(
    fusion_distribution_summary_df,
    "fusion_risk_distribution_summary.csv",
    description="Summary of fused-risk distributions by fusion OR label"
)


# Confusion matrix figure

fig_cm = plt.figure(figsize=(5.5, 4.8))
ax = fig_cm.add_subplot(111)

ConfusionMatrixDisplay(cm_fusion, display_labels=[0, 1]).plot(ax=ax, colorbar=False)
ax.set_title("Fusion Confusion Matrix (OR Label Definition)")

save_figure(
    fig_cm,
    "fusion_confusion_matrix_or.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description="Fusion confusion matrix using OR label definition"
)


# Fused-risk distribution figure

fig_dist = plt.figure(figsize=(7, 5))
ax = fig_dist.add_subplot(111)

risk_neg = fusion_df.loc[fusion_df["fusion_label_or"] == 0, "fusion_risk"].to_numpy()
risk_pos = fusion_df.loc[fusion_df["fusion_label_or"] == 1, "fusion_risk"].to_numpy()

ax.hist(risk_neg, bins=30, alpha=0.7, density=True, label="Label 0")
ax.hist(risk_pos, bins=30, alpha=0.7, density=True, label="Label 1")
ax.axvline(CONFIG["fusion_risk_threshold"], linestyle="--", label="Fusion threshold")

ax.set_xlabel("Fused Risk Score")
ax.set_ylabel("Density")
ax.set_title("Fused-Risk Distributions by OR Label")
ax.legend()
ax.grid(alpha=0.3)

save_figure(
    fig_dist,
    "fusion_risk_distribution_or.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description="Distribution of fused risk scores by OR label"
)

append_summary("Saved figure: fusion_confusion_matrix_or.png")
append_summary("Saved figure: fusion_risk_distribution_or.png")

print("Saved fusion confusion matrix and fused-risk distribution figures.")
print(fusion_distribution_summary_df)

Saved table: outputs\run_20260409_172935_seed42\tables\fusion_risk_distribution_summary.csv
Saved figure: outputs\run_20260409_172935_seed42\figures\fusion_confusion_matrix_or.png
Saved figure: outputs\run_20260409_172935_seed42\figures\fusion_risk_distribution_or.png
Saved fusion confusion matrix and fused-risk distribution figures.
     group  count  mean_fusion_risk  median_fusion_risk
0  label_0   2314          0.002714                 0.0
1  label_1  54829          0.833171                 1.0


In [ ]:

# Fusion ROC/PR curves and threshold sweep


from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)

assert "fusion_df" in globals(), "fusion_df is missing"


# Base arrays

y_fusion = fusion_df["fusion_label_or"].astype(int).to_numpy()
fusion_scores = fusion_df["fusion_risk"].astype(float).to_numpy()


# ROC and PR curves

fpr_curve, tpr_curve, roc_thresholds = roc_curve(y_fusion, fusion_scores)
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_fusion, fusion_scores)

fusion_roc_auc = roc_auc_score(y_fusion, fusion_scores)
fusion_pr_auc = average_precision_score(y_fusion, fusion_scores)

curve_summary_df = pd.DataFrame({
    "metric": ["roc_auc", "pr_auc"],
    "value": [fusion_roc_auc, fusion_pr_auc]
})

save_df(
    curve_summary_df,
    "fusion_curve_summary.csv",
    description="Fusion ROC-AUC and PR-AUC summary"
)


# Threshold sweep

threshold_grid = np.round(np.arange(0.00, 1.0001, 0.01), 2)

threshold_records = []
for thr in threshold_grid:
    y_pred_thr = (fusion_scores >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_fusion, y_pred_thr).ravel()

    rec = {
        "threshold": float(thr),
        "accuracy": float(accuracy_score(y_fusion, y_pred_thr)),
        "balanced_accuracy": float(balanced_accuracy_score(y_fusion, y_pred_thr)),
        "precision": float(precision_score(y_fusion, y_pred_thr, zero_division=0)),
        "recall": float(recall_score(y_fusion, y_pred_thr, zero_division=0)),
        "f1": float(f1_score(y_fusion, y_pred_thr, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fpr": float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
        "pred_positive_rate": float(y_pred_thr.mean())
    }
    threshold_records.append(rec)

fusion_threshold_df = pd.DataFrame(threshold_records)

save_df(
    fusion_threshold_df,
    "fusion_threshold_sweep.csv",
    description="Threshold sweep for fused risk score under OR label definition"
)


# Recommended operating points

best_f1_row = fusion_threshold_df.sort_values(
    by=["f1", "balanced_accuracy", "recall"],
    ascending=False
).iloc[0]

best_bal_acc_row = fusion_threshold_df.sort_values(
    by=["balanced_accuracy", "f1", "recall"],
    ascending=False
).iloc[0]

target_fpr = 0.01
eligible_fpr = fusion_threshold_df[fusion_threshold_df["fpr"] <= target_fpr].copy()
if len(eligible_fpr) > 0:
    best_low_fpr_row = eligible_fpr.sort_values(
        by=["recall", "f1", "balanced_accuracy"],
        ascending=False
    ).iloc[0]
else:
    best_low_fpr_row = None

recommended_rows = [
    {
        "selection_rule": "best_f1",
        **best_f1_row.to_dict()
    },
    {
        "selection_rule": "best_balanced_accuracy",
        **best_bal_acc_row.to_dict()
    }
]

if best_low_fpr_row is not None:
    recommended_rows.append({
        "selection_rule": f"best_recall_under_fpr_le_{target_fpr:.2f}",
        **best_low_fpr_row.to_dict()
    })

fusion_recommended_df = pd.DataFrame(recommended_rows)

save_df(
    fusion_recommended_df,
    "fusion_recommended_thresholds.csv",
    description="Recommended operating thresholds for fused risk score"
)


# Figure: ROC curve

fig_roc = plt.figure(figsize=(6.5, 5))
ax = fig_roc.add_subplot(111)

ax.plot(fpr_curve, tpr_curve, label=f"Fusion ROC (AUC={fusion_roc_auc:.6f})")
ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Fusion ROC Curve")
ax.legend()
ax.grid(alpha=0.3)

save_figure(
    fig_roc,
    "fusion_roc_curve_or.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description="ROC curve for fused risk score under OR label definition"
)


# Figure: PR curve

fig_pr = plt.figure(figsize=(6.5, 5))
ax = fig_pr.add_subplot(111)

ax.plot(recall_curve, precision_curve, label=f"Fusion PR (AUC={fusion_pr_auc:.6f})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Fusion Precision-Recall Curve")
ax.legend()
ax.grid(alpha=0.3)

save_figure(
    fig_pr,
    "fusion_pr_curve_or.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description="Precision-recall curve for fused risk score under OR label definition"
)


# Figure: Threshold sweep

fig_thr = plt.figure(figsize=(8, 5.5))
ax = fig_thr.add_subplot(111)

ax.plot(fusion_threshold_df["threshold"], fusion_threshold_df["balanced_accuracy"], label="Balanced Accuracy")
ax.plot(fusion_threshold_df["threshold"], fusion_threshold_df["f1"], label="F1")
ax.plot(fusion_threshold_df["threshold"], fusion_threshold_df["recall"], label="Recall")
ax.plot(fusion_threshold_df["threshold"], fusion_threshold_df["fpr"], label="FPR")
ax.axvline(CONFIG["fusion_risk_threshold"], linestyle="--", label="Current threshold")

if best_low_fpr_row is not None:
    ax.axvline(best_low_fpr_row["threshold"], linestyle=":", label=f"Best threshold (FPR≤{target_fpr:.2f})")

ax.set_xlabel("Fusion Risk Threshold")
ax.set_ylabel("Metric Value")
ax.set_title("Fusion Threshold Sweep")
ax.legend()
ax.grid(alpha=0.3)

save_figure(
    fig_thr,
    "fusion_threshold_sweep_or.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description="Threshold sweep for fused risk score under OR label definition"
)

append_summary(f"Fusion ROC-AUC: {fusion_roc_auc:.6f}")
append_summary(f"Fusion PR-AUC: {fusion_pr_auc:.6f}")
append_summary("Saved figure: fusion_roc_curve_or.png")
append_summary("Saved figure: fusion_pr_curve_or.png")
append_summary("Saved figure: fusion_threshold_sweep_or.png")

print("Saved fusion ROC, PR, and threshold-sweep outputs.")
print("\nCurve summary:")
print(curve_summary_df)

print("\nRecommended thresholds:")
print(fusion_recommended_df)

Saved table: outputs\run_20260409_172935_seed42\tables\fusion_curve_summary.csv
Saved table: outputs\run_20260409_172935_seed42\tables\fusion_threshold_sweep.csv
Saved table: outputs\run_20260409_172935_seed42\tables\fusion_recommended_thresholds.csv
Saved figure: outputs\run_20260409_172935_seed42\figures\fusion_roc_curve_or.png
Saved figure: outputs\run_20260409_172935_seed42\figures\fusion_pr_curve_or.png
Saved figure: outputs\run_20260409_172935_seed42\figures\fusion_threshold_sweep_or.png
Saved fusion ROC, PR, and threshold-sweep outputs.

Curve summary:
    metric  value
0  roc_auc    1.0
1   pr_auc    1.0

Recommended thresholds:
                  selection_rule  threshold  accuracy  balanced_accuracy  \
0                        best_f1       0.15  0.999895           0.998911   
1         best_balanced_accuracy       0.39  0.999825           0.999909   
2  best_recall_under_fpr_le_0.01       0.08  0.999790           0.997407   

   precision    recall        f1      tn    fp    

In [ ]:

# Generate scored event tables


assert "best_cyber_pipe" in globals(), "best_cyber_pipe is missing"
assert "best_video_pipe" in globals(), "best_video_pipe is missing"
assert "Xc_test" in globals() and "yc_test" in globals(), "Cyber test split missing"
assert "Xv_test" in globals() and "yv_test" in globals(), "Video test split missing"
assert "cyber_df_raw_for_fusion" in globals(), "cyber_df_raw_for_fusion is missing"
assert "video_df_raw_for_fusion" in globals(), "video_df_raw_for_fusion is missing"


# Cyber scores

cyber_scores = best_cyber_pipe.predict_proba(Xc_test)[:, 1]

cyber_scored = pd.DataFrame({
    "source_index": Xc_test.index,
    "true_label": yc_test.loc[Xc_test.index].values,
    "cyber_score": cyber_scores
})

# Recover event context from the raw cyber copy
cyber_raw_cols = []
for c in ["src_ip", "dst_ip", "src_port", "dst_port", "proto", "service", "type"]:
    if c in cyber_df_raw_for_fusion.columns:
        cyber_raw_cols.append(c)

cyber_context = cyber_df_raw_for_fusion.loc[cyber_scored["source_index"], cyber_raw_cols].reset_index(drop=True)
cyber_scored = pd.concat([cyber_scored.reset_index(drop=True), cyber_context], axis=1)

# Timestamp handling for cyber
cyber_time_col = CONFIG.get("cyber_time_col", "timestamp")
if cyber_time_col in cyber_df_raw_for_fusion.columns:
    cyber_scored["timestamp"] = pd.to_datetime(
        cyber_df_raw_for_fusion.loc[cyber_scored["source_index"], cyber_time_col].values,
        errors="coerce"
    )
    cyber_timestamp_source = "raw_timestamp_column"
else:
    cyber_scored["timestamp"] = pd.date_range(
        start="2024-01-01",
        periods=len(cyber_scored),
        freq="s"
    )
    cyber_timestamp_source = "synthetic_sequential_fallback"

# Cyber uncertainty from distance to decision boundary
cyber_scored["cyber_uncertainty"] = 1.0 - np.abs(cyber_scored["cyber_score"] - 0.5) * 2.0
cyber_scored["cyber_uncertainty"] = cyber_scored["cyber_uncertainty"].clip(0, 1)


# Video scores

video_scores = best_video_pipe.predict_proba(Xv_test)[:, 1]

video_scored = pd.DataFrame({
    "source_index": Xv_test.index,
    "true_label": yv_test.loc[Xv_test.index].values,
    "video_score": video_scores
})

video_raw_cols = []
for c in ["camera_id", "zone_id"]:
    if c in video_df_raw_for_fusion.columns:
        video_raw_cols.append(c)

video_context = video_df_raw_for_fusion.loc[video_scored["source_index"], video_raw_cols].reset_index(drop=True)
video_scored = pd.concat([video_scored.reset_index(drop=True), video_context], axis=1)

video_time_col = CONFIG.get("video_time_col", "timestamp")
if video_time_col in video_df_raw_for_fusion.columns:
    video_scored["timestamp"] = pd.to_datetime(
        video_df_raw_for_fusion.loc[video_scored["source_index"], video_time_col].values,
        errors="coerce"
    )
    video_timestamp_source = "raw_timestamp_column"
else:
    video_scored["timestamp"] = pd.date_range(
        start="2024-01-01",
        periods=len(video_scored),
        freq="s"
    )
    video_timestamp_source = "synthetic_sequential_fallback"

video_scored["video_uncertainty"] = 1.0 - np.abs(video_scored["video_score"] - 0.5) * 2.0
video_scored["video_uncertainty"] = video_scored["video_uncertainty"].clip(0, 1)


# Save outputs

save_df(
    cyber_scored,
    "cyber_scored_events.csv",
    description="Cyber test events with scores, context, timestamps, and uncertainty"
)
save_df(
    video_scored,
    "video_scored_events.csv",
    description="Video test events with scores, context, timestamps, and uncertainty"
)

scored_event_summary = {
    "active_run_id": RUN_ID,
    "selected_cyber_model": best_cyber_name,
    "selected_video_model": best_video_name,
    "cyber_scored_rows": int(len(cyber_scored)),
    "video_scored_rows": int(len(video_scored)),
    "cyber_timestamp_source": cyber_timestamp_source,
    "video_timestamp_source": video_timestamp_source
}
save_json_file(
    scored_event_summary,
    "scored_event_summary.json",
    category="fusion",
    description="Summary of scored event tables for cyber and video branches"
)

append_summary("Scored event tables created.")
append_summary(f"Cyber scored rows: {len(cyber_scored)}")
append_summary(f"Video scored rows: {len(video_scored)}")
append_summary(f"Cyber timestamp source: {cyber_timestamp_source}")
append_summary(f"Video timestamp source: {video_timestamp_source}")

print("Cyber scored events:", cyber_scored.shape)
print("Video scored events:", video_scored.shape)
print("Cyber timestamp source:", cyber_timestamp_source)
print("Video timestamp source:", video_timestamp_source)

Saved table: outputs\run_20260409_172935_seed42\tables\cyber_scored_events.csv
Saved table: outputs\run_20260409_172935_seed42\tables\video_scored_events.csv
Saved json: outputs\run_20260409_172935_seed42\others\scored_event_summary.json
Cyber scored events: (57143, 12)
Video scored events: (63313, 7)
Cyber timestamp source: synthetic_sequential_fallback
Video timestamp source: raw_timestamp_column


In [ ]:

# Time alignment and alert filtering


cyber_scored["timestamp"] = pd.to_datetime(cyber_scored["timestamp"], errors="coerce")
video_scored["timestamp"] = pd.to_datetime(video_scored["timestamp"], errors="coerce")

window = f"{CONFIG['fusion_time_window_sec']}s"

cyber_scored["time_window"] = cyber_scored["timestamp"].dt.floor(window)
video_scored["time_window"] = video_scored["timestamp"].dt.floor(window)

cyber_alerts = cyber_scored[
    cyber_scored["cyber_score"] >= CONFIG["cyber_alert_threshold"]
].copy()

video_alerts = video_scored[
    video_scored["video_score"] >= CONFIG["video_alert_threshold"]
].copy()

alignment_summary = {
    "active_run_id": RUN_ID,
    "fusion_time_window": window,
    "cyber_alert_threshold": float(CONFIG["cyber_alert_threshold"]),
    "video_alert_threshold": float(CONFIG["video_alert_threshold"]),
    "cyber_alerts_retained": int(len(cyber_alerts)),
    "video_alerts_retained": int(len(video_alerts)),
    "cyber_unique_windows": int(cyber_alerts["time_window"].nunique(dropna=True)),
    "video_unique_windows": int(video_alerts["time_window"].nunique(dropna=True))
}

save_json_file(
    alignment_summary,
    "time_alignment_summary.json",
    category="fusion",
    description="Summary of time alignment and alert filtering"
)

window_count_df = pd.DataFrame({
    "time_window": sorted(set(cyber_alerts["time_window"].dropna()).union(set(video_alerts["time_window"].dropna())))
})
window_count_df["cyber_alert_count"] = window_count_df["time_window"].map(
    cyber_alerts["time_window"].value_counts()
).fillna(0).astype(int)
window_count_df["video_alert_count"] = window_count_df["time_window"].map(
    video_alerts["time_window"].value_counts()
).fillna(0).astype(int)

save_df(
    window_count_df,
    "aligned_window_alert_counts.csv",
    description="Counts of retained cyber and video alerts by aligned time window"
)

append_summary(f"Fusion time window: {window}")
append_summary(f"Cyber alerts retained: {len(cyber_alerts)}")
append_summary(f"Video alerts retained: {len(video_alerts)}")
append_summary(f"Cyber unique windows: {alignment_summary['cyber_unique_windows']}")
append_summary(f"Video unique windows: {alignment_summary['video_unique_windows']}")

print("Cyber alerts retained:", len(cyber_alerts))
print("Video alerts retained:", len(video_alerts))
print("Cyber unique windows :", alignment_summary["cyber_unique_windows"])
print("Video unique windows :", alignment_summary["video_unique_windows"])

Saved json: outputs\run_20260409_172935_seed42\others\time_alignment_summary.json
Saved table: outputs\run_20260409_172935_seed42\tables\aligned_window_alert_counts.csv
Cyber alerts retained: 44549
Video alerts retained: 51919
Cyber unique windows : 477
Video unique windows : 45614


In [ ]:

# Window-level fusion table


cyber_window = (
    cyber_alerts.groupby("time_window")
    .agg(
        cyber_alert_count=("cyber_score", "size"),
        mean_cyber_score=("cyber_score", "mean"),
        max_cyber_score=("cyber_score", "max"),
        mean_cyber_uncertainty=("cyber_uncertainty", "mean")
    )
    .reset_index()
)

video_window = (
    video_alerts.groupby("time_window")
    .agg(
        video_alert_count=("video_score", "size"),
        mean_video_score=("video_score", "mean"),
        max_video_score=("video_score", "max"),
        mean_video_uncertainty=("video_uncertainty", "mean")
    )
    .reset_index()
)

fusion_window_df = cyber_window.merge(video_window, on="time_window", how="outer")

for col in [
    "cyber_alert_count", "mean_cyber_score", "max_cyber_score", "mean_cyber_uncertainty",
    "video_alert_count", "mean_video_score", "max_video_score", "mean_video_uncertainty"
]:
    if col in fusion_window_df.columns:
        fusion_window_df[col] = fusion_window_df[col].fillna(0)

fusion_window_df["modalities_present"] = (
    (fusion_window_df["cyber_alert_count"] > 0).astype(int) +
    (fusion_window_df["video_alert_count"] > 0).astype(int)
)

# Conservative equal-weight fusion (missing modality contributes zero)
fusion_window_df["fusion_score_raw"] = (
    0.5 * fusion_window_df["mean_cyber_score"] +
    0.5 * fusion_window_df["mean_video_score"]
)

fusion_window_df["fusion_uncertainty"] = (
    0.5 * fusion_window_df["mean_cyber_uncertainty"] +
    0.5 * fusion_window_df["mean_video_uncertainty"]
)

fusion_window_df["fusion_score"] = fusion_window_df["fusion_score_raw"] * (1.0 - fusion_window_df["fusion_uncertainty"])
fusion_window_df["fusion_score"] = fusion_window_df["fusion_score"].clip(0, 1)

# Also compute modality-normalized score for analysis
present_score_sum = (
    (fusion_window_df["cyber_alert_count"] > 0).astype(float) * fusion_window_df["mean_cyber_score"] +
    (fusion_window_df["video_alert_count"] > 0).astype(float) * fusion_window_df["mean_video_score"]
)
fusion_window_df["fusion_score_present_normalized"] = (
    present_score_sum / fusion_window_df["modalities_present"].replace(0, np.nan)
).fillna(0).clip(0, 1)

margin = CONFIG["uncertainty_margin"]
thr = CONFIG["fusion_risk_threshold"]

def assign_decision_state(score, uncertainty, modalities_present):
    if modalities_present < 2:
        return "single_modality_only"
    if uncertainty >= margin:
        if score >= thr:
            return "high_risk_uncertain"
        return "uncertain"
    if score >= thr:
        return "high_risk"
    return "low_risk"

fusion_window_df["decision_state"] = fusion_window_df.apply(
    lambda r: assign_decision_state(
        r["fusion_score"],
        r["fusion_uncertainty"],
        r["modalities_present"]
    ),
    axis=1
)

save_df(
    fusion_window_df.sort_values(by="fusion_score", ascending=False).reset_index(drop=True),
    "fusion_window_summary.csv",
    description="Window-level multimodal fusion summary"
)

window_state_counts_df = (
    fusion_window_df["decision_state"]
    .value_counts(dropna=False)
    .rename_axis("decision_state")
    .reset_index(name="count")
)
save_df(
    window_state_counts_df,
    "fusion_window_decision_state_counts.csv",
    description="Counts of fusion-window decision states"
)

append_summary(f"Fusion windows created: {len(fusion_window_df)}")
append_summary(f"Fusion windows with both modalities: {int((fusion_window_df['modalities_present'] == 2).sum())}")
append_summary(f"Fusion windows with single modality: {int((fusion_window_df['modalities_present'] == 1).sum())}")

print(f"Fusion windows created: {len(fusion_window_df)}")
print(f"Windows with both modalities : {int((fusion_window_df['modalities_present'] == 2).sum())}")
print(f"Windows with single modality : {int((fusion_window_df['modalities_present'] == 1).sum())}")
print(fusion_window_df.head())

Saved table: outputs\run_20260409_172935_seed42\tables\fusion_window_summary.csv
Saved table: outputs\run_20260409_172935_seed42\tables\fusion_window_decision_state_counts.csv
Fusion windows created: 45895
Windows with both modalities : 196
Windows with single modality : 45699
          time_window  cyber_alert_count  mean_cyber_score  max_cyber_score  \
0 2024-01-01 00:00:00               98.0          1.000000              1.0   
1 2024-01-01 00:02:00               93.0          1.000000              1.0   
2 2024-01-01 00:04:00               94.0          1.000000              1.0   
3 2024-01-01 00:06:00               93.0          0.999892              1.0   
4 2024-01-01 00:08:00               93.0          1.000000              1.0   

   mean_cyber_uncertainty  video_alert_count  mean_video_score  \
0                0.000000                0.0              0.00   
1                0.000000                0.0              0.00   
2                0.000000                0.0     

In [ ]:

# Heterogeneous temporal graph construction


assert "cyber_alerts" in globals(), "cyber_alerts is missing"
assert "video_alerts" in globals(), "video_alerts is missing"
assert "fusion_window_df" in globals(), "fusion_window_df is missing"

G = nx.Graph()

# Add cyber evidence nodes
for idx, row in cyber_alerts.iterrows():
    node_id = f"cyber_{idx}"
    G.add_node(
        node_id,
        node_type="evidence",
        modality="cyber",
        source_index=int(row["source_index"]),
        score=float(row["cyber_score"]),
        uncertainty=float(row["cyber_uncertainty"]),
        time_window=str(row["time_window"])
    )

# Add video evidence nodes
for idx, row in video_alerts.iterrows():
    node_id = f"video_{idx}"
    G.add_node(
        node_id,
        node_type="evidence",
        modality="video",
        source_index=int(row["source_index"]),
        score=float(row["video_score"]),
        uncertainty=float(row["video_uncertainty"]),
        time_window=str(row["time_window"])
    )

shared_windows = sorted(
    set(cyber_alerts["time_window"]).intersection(set(video_alerts["time_window"]))
)

fusion_both_windows = fusion_window_df.loc[
    fusion_window_df["modalities_present"] == 2, "time_window"
].astype("datetime64[ns]").tolist()

print(f"Shared windows found from alert tables : {len(shared_windows)}")
print(f"Shared windows from fusion_window_df   : {len(fusion_both_windows)}")

# Consistency check
shared_windows_set = set(shared_windows)
fusion_both_windows_set = set(fusion_both_windows)

if shared_windows_set != fusion_both_windows_set:
    print("WARNING: mismatch between shared-window sets and fusion_window_df modality_present==2")
    print(f"Only in alert intersection : {len(shared_windows_set - fusion_both_windows_set)}")
    print(f"Only in fusion_window_df   : {len(fusion_both_windows_set - shared_windows_set)}")

fusion_window_lookup = fusion_window_df.copy()
fusion_window_lookup["time_window"] = pd.to_datetime(fusion_window_lookup["time_window"], errors="coerce")
fusion_window_lookup = fusion_window_lookup.set_index("time_window").to_dict(orient="index")

for i, tw in enumerate(shared_windows):
    if i % 50 == 0:
        print(f"Processing shared window {i + 1}/{len(shared_windows)}")

    hub_id = f"hub_{tw}"

    fw = fusion_window_lookup.get(pd.to_datetime(tw), {})
    G.add_node(
        hub_id,
        node_type="hub",
        modality="hub",
        time_window=str(tw),
        fusion_score=float(fw.get("fusion_score", 0.0)),
        fusion_uncertainty=float(fw.get("fusion_uncertainty", 1.0)),
        decision_state=str(fw.get("decision_state", "unknown"))
    )

    for idx in cyber_alerts.index[cyber_alerts["time_window"] == tw]:
        G.add_edge(hub_id, f"cyber_{idx}", relation="time_window_membership")

    for idx in video_alerts.index[video_alerts["time_window"] == tw]:
        G.add_edge(hub_id, f"video_{idx}", relation="time_window_membership")


# Graph summaries

node_type_counts = {}
modality_counts = {}
decision_state_counts = {}

for _, attrs in G.nodes(data=True):
    node_type = attrs.get("node_type", "unknown")
    modality = attrs.get("modality", "unknown")
    decision_state = attrs.get("decision_state", None)

    node_type_counts[node_type] = node_type_counts.get(node_type, 0) + 1
    modality_counts[modality] = modality_counts.get(modality, 0) + 1
    if decision_state is not None:
        decision_state_counts[decision_state] = decision_state_counts.get(decision_state, 0) + 1

component_sizes = sorted([len(c) for c in nx.connected_components(G)], reverse=True)
top_component_sizes = component_sizes[:10]

graph_summary = {
    "active_run_id": RUN_ID,
    "n_nodes": int(G.number_of_nodes()),
    "n_edges": int(G.number_of_edges()),
    "shared_windows_alert_intersection": int(len(shared_windows)),
    "shared_windows_fusion_table": int(len(fusion_both_windows)),
    "node_type_counts": node_type_counts,
    "modality_counts": modality_counts,
    "hub_decision_state_counts": decision_state_counts,
    "n_connected_components": int(nx.number_connected_components(G)),
    "largest_component_size": int(component_sizes[0]) if len(component_sizes) > 0 else 0,
    "top_10_component_sizes": top_component_sizes
}

save_json_file(
    graph_summary,
    "heterogeneous_graph_summary.json",
    category="graph",
    description="Summary statistics for the heterogeneous temporal graph"
)

component_df = pd.DataFrame({
    "component_rank": np.arange(1, len(top_component_sizes) + 1),
    "component_size": top_component_sizes
})
save_df(
    component_df,
    "heterogeneous_graph_top_components.csv",
    description="Top connected component sizes in the heterogeneous temporal graph"
)

append_summary(f"Graph nodes: {G.number_of_nodes()}")
append_summary(f"Graph edges: {G.number_of_edges()}")
append_summary(f"Graph connected components: {nx.number_connected_components(G)}")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Connected components:", nx.number_connected_components(G))
print("Top component sizes:", top_component_sizes)

Shared windows found from alert tables : 196
Shared windows from fusion_window_df   : 196
Processing shared window 1/196
Processing shared window 51/196
Processing shared window 101/196
Processing shared window 151/196
Saved json: outputs\run_20260409_172935_seed42\others\heterogeneous_graph_summary.json
Saved table: outputs\run_20260409_172935_seed42\tables\heterogeneous_graph_top_components.csv
Nodes: 96664
Edges: 18609
Connected components: 78055
Top component sizes: [107, 106, 105, 105, 105, 104, 104, 104, 104, 103]


In [ ]:

# Campaign component extraction


assert "G" in globals(), "Graph G is missing"

components = list(nx.connected_components(G))
campaign_rows = []

for comp_id, comp in enumerate(components):
    sub_nodes = list(comp)
    subgraph = G.subgraph(sub_nodes)

    cyber_nodes = [n for n in sub_nodes if G.nodes[n].get("modality") == "cyber"]
    video_nodes = [n for n in sub_nodes if G.nodes[n].get("modality") == "video"]
    hub_nodes = [n for n in sub_nodes if G.nodes[n].get("modality") == "hub"]

    n_evidence_nodes = len(cyber_nodes) + len(video_nodes)

    if n_evidence_nodes < CONFIG["min_campaign_size"]:
        continue

    cyber_scores_comp = [G.nodes[n]["score"] for n in cyber_nodes] if cyber_nodes else []
    video_scores_comp = [G.nodes[n]["score"] for n in video_nodes] if video_nodes else []
    cyber_unc_comp = [G.nodes[n]["uncertainty"] for n in cyber_nodes] if cyber_nodes else []
    video_unc_comp = [G.nodes[n]["uncertainty"] for n in video_nodes] if video_nodes else []

    hub_fusion_scores = [G.nodes[n].get("fusion_score", 0.0) for n in hub_nodes] if hub_nodes else [0.0]
    hub_uncertainties = [G.nodes[n].get("fusion_uncertainty", 1.0) for n in hub_nodes] if hub_nodes else [1.0]
    decision_states = [G.nodes[n].get("decision_state", "unknown") for n in hub_nodes] if hub_nodes else ["unknown"]

    modality_pattern = (
        ("cyber" if len(cyber_nodes) > 0 else "") +
        ("+video" if len(video_nodes) > 0 else "") +
        ("+hub" if len(hub_nodes) > 0 else "")
    ).strip("+")
    if modality_pattern == "":
        modality_pattern = "unknown"

    campaign_rows.append({
        "campaign_id": int(comp_id),
        "n_nodes_total": int(len(sub_nodes)),
        "n_evidence_nodes": int(n_evidence_nodes),
        "n_edges": int(subgraph.number_of_edges()),
        "n_cyber_nodes": int(len(cyber_nodes)),
        "n_video_nodes": int(len(video_nodes)),
        "n_hub_nodes": int(len(hub_nodes)),
        "modality_pattern": modality_pattern,
        "mean_cyber_score": float(np.mean(cyber_scores_comp)) if cyber_scores_comp else 0.0,
        "mean_video_score": float(np.mean(video_scores_comp)) if video_scores_comp else 0.0,
        "mean_cyber_uncertainty": float(np.mean(cyber_unc_comp)) if cyber_unc_comp else 1.0,
        "mean_video_uncertainty": float(np.mean(video_unc_comp)) if video_unc_comp else 1.0,
        "fusion_score": float(np.mean(hub_fusion_scores)),
        "fusion_uncertainty": float(np.mean(hub_uncertainties)),
        "decision_state": max(set(decision_states), key=decision_states.count)
    })

campaign_df = pd.DataFrame(campaign_rows)

if not campaign_df.empty:
    campaign_df = campaign_df.sort_values(
        by=["fusion_score", "n_evidence_nodes"],
        ascending=[False, False]
    ).reset_index(drop=True)

save_df(
    campaign_df,
    "campaign_candidates.csv",
    description="Extracted connected campaign candidates"
)

decision_state_counts_df = (
    campaign_df["decision_state"]
    .value_counts(dropna=False)
    .rename_axis("decision_state")
    .reset_index(name="count")
)
save_df(
    decision_state_counts_df,
    "campaign_decision_state_counts.csv",
    description="Counts of extracted campaign candidates by dominant decision state"
)

modality_pattern_counts_df = (
    campaign_df["modality_pattern"]
    .value_counts(dropna=False)
    .rename_axis("modality_pattern")
    .reset_index(name="count")
)
save_df(
    modality_pattern_counts_df,
    "campaign_modality_pattern_counts.csv",
    description="Counts of extracted campaign candidates by modality composition"
)

campaign_summary = {
    "active_run_id": RUN_ID,
    "campaign_candidates_retained": int(len(campaign_df)),
    "max_campaign_size": int(campaign_df["n_evidence_nodes"].max()) if not campaign_df.empty else 0,
    "mean_campaign_size": float(campaign_df["n_evidence_nodes"].mean()) if not campaign_df.empty else 0.0,
    "high_risk_campaigns": int((campaign_df["decision_state"] == "high_risk").sum()) if not campaign_df.empty else 0
}
save_json_file(
    campaign_summary,
    "campaign_summary.json",
    category="graph",
    description="Summary of extracted campaign candidates"
)

append_summary(f"Campaign candidates retained: {len(campaign_df)}")
append_summary(f"High-risk campaigns: {campaign_summary['high_risk_campaigns']}")

print(f"Campaign candidates retained: {len(campaign_df)}")
print(campaign_df.head())

Saved table: outputs\run_20260409_172935_seed42\tables\campaign_candidates.csv
Saved table: outputs\run_20260409_172935_seed42\tables\campaign_decision_state_counts.csv
Saved table: outputs\run_20260409_172935_seed42\tables\campaign_modality_pattern_counts.csv
Saved json: outputs\run_20260409_172935_seed42\others\campaign_summary.json
Campaign candidates retained: 196
   campaign_id  n_nodes_total  n_evidence_nodes  n_edges  n_cyber_nodes  \
0         8457            105               104      104            103   
1        15066            105               104      104            103   
2         9756            104               103      103            102   
3        11431            104               103      103            102   
4        23072            104               103      103            102   

   n_video_nodes  n_hub_nodes modality_pattern  mean_cyber_score  \
0              1            1  cyber+video+hub               1.0   
1              1            1  cyber+video

In [ ]:

# Figure: Top campaign candidates


assert "campaign_df" in globals(), "campaign_df is missing"
assert not campaign_df.empty, "campaign_df is empty"

top_k = min(15, len(campaign_df))
top_campaigns_df = campaign_df.head(top_k).copy()

save_df(
    top_campaigns_df,
    "top_campaign_candidates.csv",
    description="Top campaign candidates by fusion score and evidence size"
)

fig = plt.figure(figsize=(10, 5.5))
ax = fig.add_subplot(111)

x = np.arange(len(top_campaigns_df))

ax.bar(x, top_campaigns_df["n_evidence_nodes"], label="Evidence nodes")
ax.plot(x, top_campaigns_df["fusion_score"], marker="o", label="Fusion score")

ax.set_xticks(x)
ax.set_xticklabels(top_campaigns_df["campaign_id"].astype(str), rotation=45)
ax.set_xlabel("Campaign ID")
ax.set_ylabel("Value")
ax.set_title("Top Campaign Candidates by Size and Fusion Score")
ax.legend()
ax.grid(axis="y", alpha=0.3)

save_figure(
    fig,
    "top_campaign_candidates.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description="Top campaign candidates by evidence-node count and fusion score"
)

append_summary("Saved figure: top_campaign_candidates.png")

print("Saved top campaign candidates figure.")
print(top_campaigns_df[[
    "campaign_id", "n_evidence_nodes", "n_cyber_nodes", "n_video_nodes",
    "n_hub_nodes", "fusion_score", "fusion_uncertainty", "decision_state"
]])

Saved table: outputs\run_20260409_172935_seed42\tables\top_campaign_candidates.csv
Saved figure: outputs\run_20260409_172935_seed42\figures\top_campaign_candidates.png
Saved top campaign candidates figure.
    campaign_id  n_evidence_nodes  n_cyber_nodes  n_video_nodes  n_hub_nodes  \
0          8457               104            103              1            1   
1         15066               104            103              1            1   
2          9756               103            102              1            1   
3         11431               103            102              1            1   
4         23072               103            102              1            1   
5          3696               102            101              1            1   
6         15257               102            100              2            1   
7         20731               102            101              1            1   
8          4456               101            100              1           

In [ ]:

# Figure: Visualize top campaign subgraph


assert "campaign_df" in globals(), "campaign_df is missing"
assert "G" in globals(), "Graph G is missing"
assert not campaign_df.empty, "campaign_df is empty"

# Select the top campaign
top_campaign_id = int(campaign_df.iloc[0]["campaign_id"])

components = list(nx.connected_components(G))
top_component_nodes = list(components[top_campaign_id])
top_subgraph = G.subgraph(top_component_nodes).copy()

# Save node-level summary for the top campaign
top_campaign_nodes = []
for n, attrs in top_subgraph.nodes(data=True):
    top_campaign_nodes.append({
        "node_id": n,
        "node_type": attrs.get("node_type", "unknown"),
        "modality": attrs.get("modality", "unknown"),
        "score": attrs.get("score", np.nan),
        "uncertainty": attrs.get("uncertainty", np.nan),
        "fusion_score": attrs.get("fusion_score", np.nan),
        "fusion_uncertainty": attrs.get("fusion_uncertainty", np.nan),
        "decision_state": attrs.get("decision_state", None),
        "time_window": attrs.get("time_window", None)
    })

top_campaign_nodes_df = pd.DataFrame(top_campaign_nodes)
save_df(
    top_campaign_nodes_df,
    f"top_campaign_{top_campaign_id}_nodes.csv",
    description=f"Node-level summary for top campaign {top_campaign_id}"
)

# Layout and draw
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111)

pos = nx.spring_layout(top_subgraph, seed=SEED)

cyber_nodes = [n for n, a in top_subgraph.nodes(data=True) if a.get("modality") == "cyber"]
video_nodes = [n for n, a in top_subgraph.nodes(data=True) if a.get("modality") == "video"]
hub_nodes   = [n for n, a in top_subgraph.nodes(data=True) if a.get("modality") == "hub"]

nx.draw_networkx_edges(top_subgraph, pos, ax=ax, alpha=0.35)

if len(cyber_nodes) > 0:
    nx.draw_networkx_nodes(
        top_subgraph, pos, nodelist=cyber_nodes, ax=ax,
        node_shape="o", label="Cyber evidence"
    )

if len(video_nodes) > 0:
    nx.draw_networkx_nodes(
        top_subgraph, pos, nodelist=video_nodes, ax=ax,
        node_shape="s", label="Video evidence"
    )

if len(hub_nodes) > 0:
    nx.draw_networkx_nodes(
        top_subgraph, pos, nodelist=hub_nodes, ax=ax,
        node_shape="D", label="Fusion hub"
    )

# Label only hubs and video nodes to avoid clutter
label_nodes = hub_nodes + video_nodes
label_map = {n: n for n in label_nodes}
if len(label_map) > 0:
    nx.draw_networkx_labels(top_subgraph, pos, labels=label_map, font_size=8, ax=ax)

ax.set_title(f"Top Campaign Subgraph (campaign_id={top_campaign_id})")
ax.legend()
ax.axis("off")

save_figure(
    fig,
    f"top_campaign_{top_campaign_id}_subgraph.png",
    dpi=CONFIG.get("figure_dpi", 300),
    description=f"Subgraph visualization for top campaign {top_campaign_id}"
)

append_summary(f"Saved figure: top_campaign_{top_campaign_id}_subgraph.png")

print(f"Top campaign ID: {top_campaign_id}")
print(f"Top campaign nodes: {top_subgraph.number_of_nodes()}")
print(f"Top campaign edges: {top_subgraph.number_of_edges()}")
print(top_campaign_nodes_df.head())

Saved table: outputs\run_20260409_172935_seed42\tables\top_campaign_8457_nodes.csv
Saved figure: outputs\run_20260409_172935_seed42\figures\top_campaign_8457_subgraph.png
Top campaign ID: 8457
Top campaign nodes: 105
Top campaign edges: 104
       node_id node_type modality  score  uncertainty  fusion_score  \
0  cyber_18061  evidence    cyber    1.0          0.0           NaN   
1  cyber_18092  evidence    cyber    1.0          0.0           NaN   
2  cyber_18000  evidence    cyber    1.0          0.0           NaN   
3  cyber_18048  evidence    cyber    1.0          0.0           NaN   
4  cyber_18060  evidence    cyber    1.0          0.0           NaN   

   fusion_uncertainty decision_state          time_window  
0                 NaN           None  2024-01-01 05:00:00  
1                 NaN           None  2024-01-01 05:00:00  
2                 NaN           None  2024-01-01 05:00:00  
3                 NaN           None  2024-01-01 05:00:00  
4                 NaN           

In [ ]:

# Campaign outcome summary


assert "campaign_df" in globals(), "campaign_df is missing"

if campaign_df.empty:
    campaign_summary = pd.DataFrame(columns=["decision_state", "count"])
else:
    campaign_summary = (
        campaign_df["decision_state"]
        .value_counts(dropna=False)
        .rename_axis("decision_state")
        .reset_index(name="count")
    )

save_df(
    campaign_summary,
    "campaign_decision_summary.csv",
    description="Campaign decision state summary"
)

append_summary("Campaign decision summary generated.")
print(f"Active run ID: {RUN_ID}")
print(campaign_summary)

Saved table: outputs\run_20260409_172935_seed42\tables\campaign_decision_summary.csv
Active run ID: run_20260409_172935_seed42
  decision_state  count
0      high_risk    196


In [ ]:

# Results inventory for manuscript drafting


results_inventory = [
    {
        "artifact_type": "figure",
        "filename": "video_strict_vs_full_comparison.png",
        "supports_section": "Video branch ablation results",
        "purpose": "Shows the performance gap between strict feature exclusion and the full video feature set."
    },
    {
        "artifact_type": "figure",
        "filename": "cyber_model_comparison.png",
        "supports_section": "Cyber branch results",
        "purpose": "Compares cyber branch models using balanced accuracy, ROC-AUC, PR-AUC, and FPR."
    },
    {
        "artifact_type": "figure",
        "filename": "selected_branch_confusion_matrices.png",
        "supports_section": "Branch-level model evaluation",
        "purpose": "Shows confusion matrices for the selected cyber and video models."
    },
    {
        "artifact_type": "figure",
        "filename": "selected_branch_roc_curves.png",
        "supports_section": "Branch-level model evaluation",
        "purpose": "Shows ROC curves for the selected cyber and video models."
    },
    {
        "artifact_type": "figure",
        "filename": "selected_branch_pr_curves.png",
        "supports_section": "Branch-level model evaluation",
        "purpose": "Shows precision-recall curves for the selected cyber and video models."
    },
    {
        "artifact_type": "figure",
        "filename": "fusion_confusion_matrix_or.png",
        "supports_section": "Fusion results",
        "purpose": "Shows the confusion matrix for the fusion model under OR-label evaluation."
    },
    {
        "artifact_type": "figure",
        "filename": "fusion_risk_distribution_or.png",
        "supports_section": "Fusion results",
        "purpose": "Shows fused-risk score distributions for negative and positive fusion labels."
    },
    {
        "artifact_type": "figure",
        "filename": "fusion_roc_curve_or.png",
        "supports_section": "Fusion results",
        "purpose": "Shows ROC curve for the fused risk score."
    },
    {
        "artifact_type": "figure",
        "filename": "fusion_pr_curve_or.png",
        "supports_section": "Fusion results",
        "purpose": "Shows precision-recall curve for the fused risk score."
    },
    {
        "artifact_type": "figure",
        "filename": "fusion_threshold_sweep_or.png",
        "supports_section": "Fusion threshold analysis",
        "purpose": "Shows how balanced accuracy, F1, recall, and FPR vary with threshold."
    },
    {
        "artifact_type": "figure",
        "filename": "top_campaign_candidates.png",
        "supports_section": "Campaign reconstruction results",
        "purpose": "Summarizes top campaign candidates by size and fusion score."
    },
    {
        "artifact_type": "figure",
        "filename": "top_campaign_8457_subgraph.png",
        "supports_section": "Campaign reconstruction results",
        "purpose": "Visualizes the subgraph structure of the top extracted campaign."
    },
    {
        "artifact_type": "table",
        "filename": "cyber_model_results.csv",
        "supports_section": "Cyber branch results",
        "purpose": "Detailed cyber model comparison metrics."
    },
    {
        "artifact_type": "table",
        "filename": "video_model_results_strict.csv",
        "supports_section": "Video branch ablation results",
        "purpose": "Strict-ablation video model comparison metrics."
    },
    {
        "artifact_type": "table",
        "filename": "video_model_results_full.csv",
        "supports_section": "Video branch results",
        "purpose": "Full-feature video model comparison metrics."
    },
    {
        "artifact_type": "table",
        "filename": "fusion_metrics_or.csv",
        "supports_section": "Fusion results",
        "purpose": "Fusion-level evaluation metrics under OR-label definition."
    },
    {
        "artifact_type": "table",
        "filename": "fusion_recommended_thresholds.csv",
        "supports_section": "Fusion threshold analysis",
        "purpose": "Recommended fusion thresholds and associated metrics."
    },
    {
        "artifact_type": "table",
        "filename": "campaign_candidates.csv",
        "supports_section": "Campaign reconstruction results",
        "purpose": "Extracted campaign-level candidates with structural and scoring summaries."
    },
    {
        "artifact_type": "table",
        "filename": "campaign_decision_summary.csv",
        "supports_section": "Campaign reconstruction results",
        "purpose": "Summary of dominant campaign decision states."
    }
]

results_inventory_df = pd.DataFrame(results_inventory)

save_df(
    results_inventory_df,
    "results_inventory.csv",
    description="Inventory of key saved figures and tables for manuscript drafting"
)

append_summary("Results inventory created.")
print(results_inventory_df)

Saved table: outputs\run_20260409_172935_seed42\tables\results_inventory.csv
   artifact_type                                filename  \
0         figure     video_strict_vs_full_comparison.png   
1         figure              cyber_model_comparison.png   
2         figure  selected_branch_confusion_matrices.png   
3         figure          selected_branch_roc_curves.png   
4         figure           selected_branch_pr_curves.png   
5         figure          fusion_confusion_matrix_or.png   
6         figure         fusion_risk_distribution_or.png   
7         figure                 fusion_roc_curve_or.png   
8         figure                  fusion_pr_curve_or.png   
9         figure           fusion_threshold_sweep_or.png   
10        figure             top_campaign_candidates.png   
11        figure          top_campaign_8457_subgraph.png   
12         table                 cyber_model_results.csv   
13         table          video_model_results_strict.csv   
14         table       

In [ ]:

# Fusion vs single-modality summary


assert "cyber_alerts" in globals(), "cyber_alerts is missing"
assert "video_alerts" in globals(), "video_alerts is missing"
assert "fusion_window_df" in globals(), "fusion_window_df is missing"
assert "campaign_df" in globals(), "campaign_df is missing"

fusion_vs_single_df = pd.DataFrame([
    {
        "method": "Cyber only",
        "selected_model": best_cyber_name,
        "analysis_unit": "event",
        "threshold": float(CONFIG["cyber_alert_threshold"]),
        "events_or_windows_detected": int(len(cyber_alerts)),
        "campaigns_detected": 0,
        "note": "Cyber alert events above the cyber decision threshold."
    },
    {
        "method": "Video only",
        "selected_model": best_video_name,
        "analysis_unit": "event",
        "threshold": float(CONFIG["video_alert_threshold"]),
        "events_or_windows_detected": int(len(video_alerts)),
        "campaigns_detected": 0,
        "note": "Video alert events above the video decision threshold."
    },
    {
        "method": "Fusion",
        "selected_model": f"{best_cyber_name}+{best_video_name}",
        "analysis_unit": "window",
        "threshold": float(CONFIG["fusion_risk_threshold"]),
        "events_or_windows_detected": int(len(fusion_window_df)),
        "campaigns_detected": int(len(campaign_df)),
        "note": "Window-level fused detections and connected campaign candidates."
    }
])

save_df(
    fusion_vs_single_df,
    "fusion_vs_single_modality_summary.csv",
    description="Fusion versus single-modality detection summary"
)

append_summary("Fusion versus single-modality summary saved.")
print(fusion_vs_single_df)

Saved table: outputs\run_20260409_172935_seed42\tables\fusion_vs_single_modality_summary.csv
       method           selected_model analysis_unit  threshold  \
0  Cyber only               ExtraTrees         event       0.60   
1  Video only             RandomForest         event       0.60   
2      Fusion  ExtraTrees+RandomForest        window       0.65   

   events_or_windows_detected  campaigns_detected  \
0                       44549                   0   
1                       51919                   0   
2                       45895                 196   

                                                note  
0  Cyber alert events above the cyber decision th...  
1  Video alert events above the video decision th...  
2  Window-level fused detections and connected ca...  


In [ ]:

# SHAP explainability for cyber branch


print("Preparing SHAP analysis for cyber branch...")

assert "best_cyber_name" in globals(), "best_cyber_name is missing"
assert "best_cyber_pipe" in globals(), "best_cyber_pipe is missing"
assert "Xc_train_use" in globals(), "Xc_train_use is missing"
assert "Xc_test_use" in globals(), "Xc_test_use is missing"

if best_cyber_name in ["RandomForest", "ExtraTrees"]:
    try:
        # ----------------------------------------------------
        # Sample data
        # ----------------------------------------------------
        n_bg = min(200, len(Xc_train_use))
        n_sample = min(100, len(Xc_test_use))

        Xc_bg = Xc_train_use.sample(n_bg, random_state=SEED)
        Xc_sample = Xc_test_use.sample(n_sample, random_state=SEED)

        # ----------------------------------------------------
        # Transform using the fitted preprocessor
        # ----------------------------------------------------
        preprocessor = best_cyber_pipe.named_steps["preprocessor"]
        model = best_cyber_pipe.named_steps["model"]

        Xc_bg_proc = preprocessor.transform(Xc_bg)
        Xc_sample_proc = preprocessor.transform(Xc_sample)

        bg_was_sparse = sparse.issparse(Xc_bg_proc)
        sample_was_sparse = sparse.issparse(Xc_sample_proc)

        if bg_was_sparse:
            Xc_bg_proc = Xc_bg_proc.toarray()
        if sample_was_sparse:
            Xc_sample_proc = Xc_sample_proc.toarray()

        Xc_bg_proc = np.asarray(Xc_bg_proc, dtype=np.float32)
        Xc_sample_proc = np.asarray(Xc_sample_proc, dtype=np.float32)

        # ----------------------------------------------------
        # Feature names
        # ----------------------------------------------------
        try:
            feature_names = list(preprocessor.get_feature_names_out())
        except Exception:
            feature_names = [f"feature_{i}" for i in range(Xc_sample_proc.shape[1])]

        # ----------------------------------------------------
        # SHAP explainer
        # ----------------------------------------------------
        explainer = shap.TreeExplainer(
            model,
            data=Xc_bg_proc,
            feature_perturbation="interventional"
        )

        shap_values = explainer.shap_values(
            Xc_sample_proc,
            check_additivity=False
        )

        # ----------------------------------------------------
        # Normalize SHAP output to a 2D array: (samples, features)
        # ----------------------------------------------------
        if isinstance(shap_values, list):
            shap_arr = shap_values[1] if len(shap_values) > 1 else shap_values[0]
            shap_arr = np.asarray(shap_arr, dtype=np.float32)

        elif hasattr(shap_values, "values"):
            shap_arr = np.asarray(shap_values.values, dtype=np.float32)

        else:
            shap_arr = np.asarray(shap_values, dtype=np.float32)

        # Handle possible 3D outputs
        if shap_arr.ndim == 3:
            # Common cases:
            # (samples, features, classes) -> select positive class
            # (classes, samples, features) -> select positive class
            if shap_arr.shape[2] == 2 and shap_arr.shape[0] == Xc_sample_proc.shape[0]:
                shap_arr = shap_arr[:, :, 1]
            elif shap_arr.shape[0] == 2 and shap_arr.shape[1] == Xc_sample_proc.shape[0]:
                shap_arr = shap_arr[1, :, :]
            else:
                # Fallback: average across the smallest axis that looks like class axis
                class_axis = int(np.argmin(shap_arr.shape))
                shap_arr = np.mean(shap_arr, axis=class_axis)

        # Final safety checks
        shap_arr = np.asarray(shap_arr, dtype=np.float32)

        if shap_arr.ndim != 2:
            raise ValueError(f"Normalized SHAP array must be 2D, got shape {shap_arr.shape}")

        if shap_arr.shape[1] != len(feature_names):
            raise ValueError(
                f"Feature-name length mismatch: shap features={shap_arr.shape[1]}, "
                f"feature_names={len(feature_names)}"
            )

        # ----------------------------------------------------
        # Save mean absolute SHAP importance
        # ----------------------------------------------------
        mean_abs_shap = np.mean(np.abs(shap_arr), axis=0).ravel()

        shap_importance_df = pd.DataFrame({
            "feature": feature_names,
            "mean_abs_shap": mean_abs_shap
        }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

        save_df(
            shap_importance_df,
            "cyber_shap_feature_importance.csv",
            description="Mean absolute SHAP importance for the selected cyber model"
        )

        # ----------------------------------------------------
        # Summary plot
        # ----------------------------------------------------
        plt.figure(figsize=(10, 6))
        shap.summary_plot(
            shap_arr,
            Xc_sample_proc,
            feature_names=feature_names,
            show=False
        )
        fig = plt.gcf()

        save_figure(
            fig,
            "cyber_shap_summary.png",
            dpi=CONFIG.get("figure_dpi", 300),
            description="SHAP summary plot for the selected cyber model"
        )

        shap_summary = {
            "active_run_id": RUN_ID,
            "selected_cyber_model": best_cyber_name,
            "background_samples": int(n_bg),
            "explained_samples": int(n_sample),
            "n_transformed_features": int(Xc_sample_proc.shape[1]),
            "normalized_shap_shape": list(shap_arr.shape),
            "background_was_sparse": bool(bg_was_sparse),
            "sample_was_sparse": bool(sample_was_sparse)
        }

        save_json_file(
            shap_summary,
            "cyber_shap_summary.json",
            category="explainability",
            description="Summary metadata for cyber SHAP analysis"
        )

        append_summary("SHAP analysis completed for cyber branch.")
        append_summary("Saved figure: cyber_shap_summary.png")
        print("SHAP summary saved successfully.")
        print("Normalized SHAP shape:", shap_arr.shape)
        print(shap_importance_df.head(15))

    except Exception as e:
        print(f"SHAP analysis failed: {e}")
        append_summary(f"SHAP analysis failed: {e}")

else:
    print("SHAP is configured here for tree-based cyber models only.")

Preparing SHAP analysis for cyber branch...
Saved table: outputs\run_20260409_172935_seed42\tables\cyber_shap_feature_importance.csv


C:\Users\amala\AppData\Local\Temp\ipykernel_30588\3130312010.py:124: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved figure: outputs\run_20260409_172935_seed42\figures\cyber_shap_summary.png
Saved json: outputs\run_20260409_172935_seed42\others\cyber_shap_summary.json
SHAP summary saved successfully.
Normalized SHAP shape: (100, 888)
                feature  mean_abs_shap
0         num__dst_port       0.042574
1        cat__proto_tcp       0.042210
2        cat__proto_udp       0.038579
3      cat__service_dns       0.036871
4   cat__dns_rejected_F       0.024693
5   cat__conn_state_REJ       0.023497
6    cat__conn_state_S0       0.023149
7         num__src_port       0.022243
8         cat__dns_RA_F       0.021577
9     cat__service_http       0.016290
10        num__src_pkts       0.014382
11       cat__service_-       0.010698
12    num__src_ip_bytes       0.010543
13        num__dst_pkts       0.009123
14     cat__dns_query_-       0.008301


In [ ]:

# Structured campaign reporting


assert "campaign_df" in globals(), "campaign_df is missing"

report_rows = []

if not campaign_df.empty:
    ranked_campaign_df = campaign_df.reset_index(drop=True).copy()
    ranked_campaign_df["campaign_rank"] = np.arange(1, len(ranked_campaign_df) + 1)

    for _, row in ranked_campaign_df.iterrows():
        report_rows.append({
            "campaign_rank": int(row["campaign_rank"]),
            "campaign_id": int(row["campaign_id"]),
            "decision_state": row["decision_state"],
            "fusion_score": round(float(row["fusion_score"]), 6),
            "fusion_uncertainty": round(float(row["fusion_uncertainty"]), 6),
            "n_nodes_total": int(row["n_nodes_total"]),
            "n_evidence_nodes": int(row["n_evidence_nodes"]),
            "n_cyber_nodes": int(row["n_cyber_nodes"]),
            "n_video_nodes": int(row["n_video_nodes"]),
            "n_hub_nodes": int(row["n_hub_nodes"]),
            "modality_pattern": row.get("modality_pattern", "unknown"),
            "summary_text": (
                f"Campaign {int(row['campaign_id'])} ranks #{int(row['campaign_rank'])} and contains "
                f"{int(row['n_evidence_nodes'])} evidence nodes "
                f"({int(row['n_cyber_nodes'])} cyber, {int(row['n_video_nodes'])} video), "
                f"fusion score {float(row['fusion_score']):.3f}, "
                f"fusion uncertainty {float(row['fusion_uncertainty']):.3f}, "
                f"decision state '{row['decision_state']}'."
            )
        })

report_df = pd.DataFrame(report_rows)

save_df(
    report_df,
    "structured_campaign_reports.csv",
    description="Structured campaign-level narrative summaries"
)

append_summary("Structured campaign reports generated.")
print(f"Structured campaign reports: {report_df.shape}")
print(report_df.head())

Saved table: outputs\run_20260409_172935_seed42\tables\structured_campaign_reports.csv
Structured campaign reports: (196, 12)
   campaign_rank  campaign_id decision_state  fusion_score  \
0              1         8457      high_risk           1.0   
1              2        15066      high_risk           1.0   
2              3         9756      high_risk           1.0   
3              4        11431      high_risk           1.0   
4              5        23072      high_risk           1.0   

   fusion_uncertainty  n_nodes_total  n_evidence_nodes  n_cyber_nodes  \
0                 0.0            105               104            103   
1                 0.0            105               104            103   
2                 0.0            104               103            102   
3                 0.0            104               103            102   
4                 0.0            104               103            102   

   n_video_nodes  n_hub_nodes modality_pattern  \
0           